# NeuraHack 2026 – ScamShield BD — Declaration

## 1. Libraries
The preinstalled Kaggle image is used. `sentence-transformers` is installed only if the image lacks it.

| Library | Version |
|---|---|
| python | Kaggle image default (printed by the setup cell) |
| numpy | 2.1.3 |
| pandas | 2.3.3 |
| scikit-learn | 1.6.1 |
| scipy | 1.16.3 |
| torch | 2.11.0 |
| transformers | 5.20.2 (reference run) |
| sentence-transformers | 5.7.0 |

The setup cell (section 1) prints python, numpy, pandas, scikit-learn, scipy, torch and transformers; cell B1 prints sentence-transformers.

## 2. External resources

| Resource | Type | Source | License | Use |
|---|---|---|---|---|
| `xlm-roberta-base` (`FacebookAI/xlm-roberta-base`) | transformer | Hugging Face Hub | MIT | Fine-tuned here (multi-task head) |
| `intfloat/multilingual-e5-large` | embedding model | Hugging Face Hub | MIT | Frozen sentence embeddings |
| `BAAI/bge-m3` | embedding model | Hugging Face Hub | MIT | Frozen sentence embeddings |
| `sentence-transformers/LaBSE` | embedding model | Hugging Face Hub | Apache-2.0 | Frozen sentence embeddings |
| `sentence-transformers` | library | PyPI | Apache-2.0 | Encoding with the three embedding models |
| Competition data (`sample.csv`, `test.csv`, `sample_submission.csv`) | data | Kaggle competition | Organizer | `sample.csv` for training/validation; `test.csv` for inference only |

* **APIs:** none. No hosted LLM or inference API is called, so `.env` is empty.
* **External labeled datasets:** none. The synthetic training rows (about 12k) are generated by code in this notebook (section 2) from hand-written templates. No test text or test label is used to write or fill them.
* **Public code:** none beyond library documentation. AI coding assistants helped write the code.

## 3. Method and validation
* **No test text is used for training** (Rulebook v1.3, Section 5, item 3). There is no pseudo-labeling, self-training or domain adaptation. Every model learns only from `sample.csv` and the synthetic rows; `test.csv` is used for inference only.
* **Text normalization.** Unicode NFKC; zero-width characters removed; Cyrillic/Greek lookalikes mapped to Latin; Bangla digits mapped to ASCII; defanged links repaired (`[.]`, `hxxp`); repeated characters squeezed; in-word leetspeak undone (Part A only); lowercased. The embedding family uses its own `clean()`, which also turns `|||` into a newline.
* **Synthetic data (Part A).** About 12,000 rows from templates with random slots. They cover all 6 classes in English, Banglish and Bangla, conversations, group chats and hard legitimate cases. 35% of synthetic rows are corrupted with realistic noise; the 86 real rows stay clean and are upweighted ×6.
  * **v3 additions:** templates for the error types seen in out-of-fold predictions on `sample.csv`: legitimate telecom, courier, HR, interview and institution notices; non-scam real-estate and service promotions; prize messages that ask for an OTP (class 1); shopping-task earning schemes (class 3). They were written from the labeled rows and general knowledge, without reading test text.
* **Model families.**
  * `tfidf`: TF-IDF (char 2–5 + word 1–2) with logistic-regression heads for label, action and 6 one-vs-rest techniques.
  * `xlmr`: fine-tuned `xlm-roberta-base` (lr 3e-5, batch 32, 3 epochs, max length 192, fp16) with a multi-task head (label softmax, action softmax, 6 technique sigmoids). The final test predictions average 2 seeds (42, 43).
  * `emb`: frozen e5-large, bge-m3 and LaBSE embeddings, each with logistic-regression heads for A, B and C, combined with fixed weights 0.367 / 0.417 / 0.217. These came from a CV grid on the same shared folds during development (best 0.4 / 0.5 / 0.1, shrunk 50% toward equal).
* **Final ensemble (Part C).** It combines `emb`, `tfidf` and `xlmr` (plus `ft` only if this notebook writes its files). Weights and the technique threshold are chosen by grid search on out-of-fold predictions, then shrunk 50% toward equal weights. The last cell also scores the TF-IDF + XLM-R blend from Part A on the same out-of-fold predictions and writes whichever of the two has the higher CV proxy to `submission.csv`; both are saved as `sub_<name>.csv`.
* **Post-rules from the labeling guide.** Labels 0 and 5 get `NONE` and action 0. Labels 1–4 get at least one technique and an action other than 0.
* **Confidence.** A temperature on the Task A probabilities, chosen by Brier score on out-of-fold predictions.
* **Validation.** `RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)` over the 86 real rows, shared by all families. OOF probabilities are averaged over repeats; XLM-R uses the first repeat only, for time. Synthetic rows are only ever in training. Synthetic templates that resemble a validation row (char cosine > 0.35) are removed from that split's training. Each split also scores a corrupted copy of its validation rows.
* **Generalization analysis.** Error analysis is in section 7 and Part B3, and the **Review** section at the end summarizes it. The composite score printed here is a proxy for the official metric.

## 4. Seeds and reproduction
* **Seeds.** `SEED = 42`: Python `random`, NumPy, torch and CUDA (cuDNN deterministic), and the CV `random_state`. The XLM-R test predictions use seeds 42 and 43.
* **Run.** Kaggle, GPU T4, Internet ON (to download the four models), attach the competition data, then **Save Version → Save & Run All**. Runtime is about 1.8 h (Part A about 1 h, embeddings about 45 min) on a T4.
* **Output.** `/kaggle/working/submission.csv`. The last cell overwrites the Part A submission with the candidate it chose (see section 3).

## 5. API steps and costs
None. No API is called, no key is used, `.env` is empty, and the cost is 0.

# Part A — v2: synthetic data + TF-IDF + XLM-R fine-tune (Anadi)
Saves `oof_/test_{tfidf,xlmr,v2ens}_{A,B,C}.npy` on the shared folds.

## 1. Setup, configuration, data loading, normalization

In [ ]:
import os, re, glob, json, time, math, random, unicodedata, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# ---------------- configuration ----------------
MODEL_NAME   = os.environ.get("SS_MODEL", "xlm-roberta-base")   # try "xlm-roberta-large" on a P100 / A100 if time allows
N_SYNTH      = int(os.environ.get("SS_N_SYNTH", 12000))         # synthetic training rows
NOISE_FRAC   = 0.35                                             # share of synthetic rows that get text corruption
REAL_UP      = 6                                                # copies of each real labeled row in training (upweighting)
REAL_NOISE_P = 0.0                                              # chance an extra real copy is noised (0 = clean; best in ablation)
EPOCHS       = float(os.environ.get("SS_EPOCHS", 3))
LR           = float(os.environ.get("SS_LR", 3e-5))
FINAL_SEEDS  = int(os.environ.get("SS_FINAL_SEEDS", 2))       # XLM-R seeds averaged for the final test predictions
BATCH        = int(os.environ.get("SS_BATCH", 32))
MAX_LEN      = 192
N_FOLDS      = 5
XLMR_REPEATS = int(os.environ.get("SS_XLMR_REPEATS", 1))       # repeats of the shared 5-fold split used for XLM-R CV (1 = 5 fits)
RUN_CV       = os.environ.get("SS_RUN_CV", "1") == "1"          # cross-validation of XLM-R (TF-IDF CV always runs; it is cheap)
USE_XLMR     = os.environ.get("SS_USE_XLMR", "1") == "1"
TEST_NROWS   = int(os.environ["SS_TEST_NROWS"]) if os.environ.get("SS_TEST_NROWS") else None   # None = all rows (local debugging only)

hits = glob.glob("/kaggle/input/**/sample.csv", recursive=True) + glob.glob(os.path.join(os.environ.get("SS_DATA_DIR", "."), "sample.csv"))
DATA_DIR = os.path.dirname(hits[0])
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("SS_OUT_DIR", ".")
train = pd.read_csv(os.path.join(DATA_DIR, "sample.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), nrows=TEST_NROWS)
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"), nrows=TEST_NROWS, keep_default_na=False)
test["text"] = test["text"].fillna("").astype(str)
print("DATA_DIR:", DATA_DIR, "| train", train.shape, "| test", test.shape)

import sys, sklearn, scipy
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)
try:
    import torch, transformers
    print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except ImportError:
    USE_XLMR = False; print("torch/transformers not available -> TF-IDF only")

TECH = ["T1", "T2", "T3", "T4", "T5", "T6"]
def tech_to_vec(s):
    s = set(str(s).split("|")) if s and s != "NONE" else set()
    return [int(t in s) for t in TECH]

# ---------------- normalization ----------------
LOOK = str.maketrans({"а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ј": "j", "ѕ": "s", "һ": "h", "ԁ": "d",
                      "ԛ": "q", "ѵ": "v", "А": "a", "Е": "e", "О": "o", "Р": "p", "С": "c", "Х": "x", "І": "i", "Т": "t", "В": "b", "М": "m",
                      "Н": "h", "К": "k", "α": "a", "ο": "o", "ρ": "p", "ν": "v", "ι": "i", "τ": "t", "κ": "k", "ε": "e"})
ZW = re.compile("[​‌‎‏⁠﻿­]")   # keep ‍ (needed inside Bangla conjuncts)
BN_DIG = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
LEET_MAP = str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t", "@": "a", "$": "s"})
REPEAT = re.compile(r"([a-zA-Z])\1{2,}")
TOKEN = re.compile(r"[A-Za-z0-9@$]+")
def _unleet(m):
    w = m.group(0)
    letters = sum(c.isalpha() for c in w); digits = len(w) - letters
    if letters >= 2 and 0 < digits <= letters and not w[0].isdigit():
        return w.translate(LEET_MAP)
    return w
def norm(t):
    t = unicodedata.normalize("NFKC", str(t))
    t = ZW.sub("", t).translate(LOOK).translate(BN_DIG)
    t = t.replace("[.]", ".").replace("[]", ".").replace("hxxp", "http")
    t = REPEAT.sub(r"\1", t)
    t = TOKEN.sub(_unleet, t)
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t if t else "empty"

for s in ["Amі agnt, bhulnumber-e taka pathaisi. Ferot dіn,PIN ta bolen.", "mnbvcx Your bKash PIIIN will expire Rpely witttth currentP1N",
          "Notice: yoru accunt WiLL be blоcked O7herwi53 pl3a5e hxxps://shwapno-login89[]vip"]:
    print(repr(s), "->", repr(norm(s)))

train["y"] = train["label"].astype(int); train["act"] = train["action"].astype(int)
train["n"] = train["text"].map(norm)
test["n"] = test["text"].map(norm)
def lang_of(t):
    if re.search(r"[ঀ-৿]", t): return "bn"
    bl = {"apnar", "apni", "korun", "kore", "taka", "hobe", "jonno", "ache", "ase", "nai", "din", "den", "pathan", "bhai", "ami", "ekta", "theke", "korte", "na", "kal"}
    return "banglish" if len(set(re.findall(r"[a-z]+", t.lower())) & bl) >= 1 else "en"
train["lang"] = train["text"].map(lang_of)
print("train langs:", train.lang.value_counts().to_dict())

## 2. Synthetic training data (generated programmatically, reproducible with SEED)
`sample.csv` has only 86 labeled rows, so we generate thousands of extra labeled rows from hand-written
templates with random slots: brands, amounts, links, phones, names. No test text or test label is used.
Coverage: all 6 classes in English, Banglish and Bangla script; single messages and multi-turn conversations
with various speaker markers; boilerplate wrappers; group chats; hard legitimate examples, including people
quoting a scam to warn someone, which is class 0; and realistic noise (typos, lookalike characters, zero-width
characters, missing spaces, leetspeak, random caps, emojis, noise strings).
Every template also carries its Task B techniques and Task C action, following the labeling guide.

In [ ]:
import random, re, itertools

def make_synth(n_rows=12000, seed=SEED):
    R = random.Random(seed)
    pick = R.choice
    MFS = ["bKash", "Nagad", "Rocket", "Upay", "Tap", "mCash", "SureCash", "OK Wallet", "Pathao Pay", "CellFin", "iPay", "Dmoney"]
    BANK = ["DBBL", "Dutch-Bangla Bank", "BRAC Bank", "City Bank", "EBL", "Islami Bank", "Sonali Bank", "Pubali Bank",
            "Mutual Trust Bank", "UCB", "Prime Bank", "AB Bank", "Standard Chartered", "IFIC Bank", "Southeast Bank", "Janata Bank"]
    COURIER = ["Sundarban Courier", "SA Paribahan", "Pathao Courier", "RedX", "Steadfast", "eCourier", "Paperfly", "DHL",
               "FedEx", "Karatoa", "Janani Courier", "Delivery Tiger", "Aramex", "Bangladesh Post", "Sheba Delivery", "UPS"]
    TELCO = ["GP", "Grameenphone", "Robi", "Banglalink", "Teletalk", "Airtel"]
    SHOP = ["Daraz", "Chaldal", "Shwapno", "Aarong", "Pickaboo", "Foodpanda", "Pathao Food", "Shohoz", "Bikroy", "Othoba",
            "Star Tech", "Ryans", "Walton", "Bata", "Apex", "Yellow", "Sailor", "Agora", "Meena Bazar", "Rokomari", "Pizza Hut", "KFC"]
    COMPANY = ["Unilever Bangladesh", "BRAC", "Grameenphone", "Walton", "Square Group", "Pran-RFL", "ACI", "Bashundhara Group",
               "Aarong", "Akij Group", "BEXIMCO", "Robi Axiata", "Daraz", "Shwapno", "Star Tech"]
    GOV = ["Bangladesh Bank", "BTRC", "NBR", "Election Commission", "PM Education Fund", "Ministry of Social Welfare", "Bangladesh Police"]
    UTIL = ["DESCO", "DPDC", "BREB", "Titas Gas", "WASA", "Karnaphuli Gas"]
    NAME = ["Rahim", "Karim", "Sumaiya", "Nusrat", "Tanvir", "Farhan", "Lisa", "Rafi", "Mitu", "Arif", "Jamal", "Sabbir", "Tania",
            "Imran", "Anika", "Rakib", "Sadia", "Hasan", "Mim", "Shakil", "Ayesha", "Fahim", "Nadia", "Riya"]
    FNAME = ["James", "Sarah", "David", "Michael", "Emma", "Lee", "Anna", "John", "Maria"]
    COUNTRY = ["China", "Malaysia", "UK", "USA", "Dubai", "Saudi Arabia", "Singapore", "Canada", "Italy", "Japan", "Qatar"]
    BN_BRAND = {"bKash": "বিকাশ", "Nagad": "নগদ", "Rocket": "রকেট", "Upay": "উপায়", "Grameenphone": "গ্রামীণফোন",
                "Robi": "রবি", "Banglalink": "বাংলালিংক", "Daraz": "দারাজ", "Sundarban Courier": "সুন্দরবন কুরিয়ার"}
    TLD = ["top", "xyz", "site", "online", "vip", "link", "click", "info", "cc", "icu", "live", "shop", "net", "buzz"]
    WORD = ["claim", "gift", "bonus", "verify", "update", "kyc", "reward", "login", "help", "secure", "pay", "track", "offer", "win"]
    BN_DIG = str.maketrans("0123456789", "০১২৩৪৫৬৭৮৯")

    def rnd_id(k, chars="abcdefghijklmnopqrstuvwxyz0123456789"): return "".join(R.choice(chars) for _ in range(k))
    def money(v):
        s = f"{v:,}"
        return pick([f"Tk {s}", f"Tk{s}", f"{s} taka", f"BDT {s}", f"৳{s}", f"Tk {v}", f"tk {s}", f"{s} tk"])
    def bn_money(v): return pick([f"{v:,} টাকা", f"৳{v:,}"]).translate(BN_DIG)
    def amt(): return money(pick([100, 150, 200, 250, 300, 400, 450, 500, 700, 800, 1000, 1200, 1500, 2000, 2500, 3000, 3500, 4000, 5000, 7000, 8500, 10000, 15000, 25000]))
    def fee(): return money(pick([25, 30, 40, 45, 50, 60, 99, 100, 120, 150, 200, 250, 300, 350, 500, 750, 1000, 1500, 2000, 3500]))
    def big(): return money(pick([5000, 10000, 20000, 50000, 100000, 200000, 500000, 1000000]))
    def phone(): return pick(["01XXXXXXXXX", f"+880 01{R.randint(3,9)}XXXXXX{R.randint(10,99)}", f"01{R.randint(3,9)}XXXXXX{R.randint(10,99)}",
                              f"01{R.randint(3,9)}{R.randint(10000000,99999999)}"[:11].replace(str(R.randint(0,9)), "X", 3)])
    def scam_link(b=None):
        b = (b or pick(MFS + COURIER + SHOP + TELCO + BANK)).lower().replace(" ", "")
        dom = pick([f"{b}-{pick(WORD)}", f"{pick(WORD)}-{b}", f"{b}{pick(WORD)}{R.randint(1,99)}", f"bd-{pick(WORD)}", f"{pick(WORD)}{pick(WORD)}bd"])
        dot = pick(["[.]", "[.]", "[]", "."])
        path = pick(["", "/" + pick(WORD), "/" + rnd_id(7), "/app.apk", "/login"])
        if R.random() < 0.15: return f"bit{dot}ly/{rnd_id(6)}"
        if R.random() < 0.1: return f"hxxps://t{dot}me/{pick(WORD)}_{pick(['bd','vip','earn','group'])}"
        return f"{pick(['hxxps://', 'hxxp://', 'hxxps://', ''])}{dom}{dot}{pick(TLD)}{path}"
    def legit_link(b=None):
        b = (b or pick(MFS + SHOP + TELCO + BANK)).lower().replace(" ", "").replace("-", "")
        dot = pick(["[.]", "[.]", "[]"])
        return f"https://{b}{dot}com{dot}bd/{pick(['track', 'offers', 'app', 'pay', 'help', 'account'])}"
    SLOT = {
        "mfs": lambda: pick(MFS), "bank": lambda: pick(BANK), "courier": lambda: pick(COURIER), "telco": lambda: pick(TELCO),
        "shop": lambda: pick(SHOP), "company": lambda: pick(COMPANY), "gov": lambda: pick(GOV), "util": lambda: pick(UTIL),
        "org": lambda: pick(MFS + BANK), "brand": lambda: pick(MFS + BANK + SHOP + TELCO), "name": lambda: pick(NAME),
        "fname": lambda: pick(FNAME), "country": lambda: pick(COUNTRY), "amt": amt, "fee": fee, "big": big, "phone": phone,
        "link": scam_link, "glink": legit_link, "code": lambda: str(R.randint(100000, 999999)), "code4": lambda: str(R.randint(1000, 9999)),
        "trx": lambda: rnd_id(10, "ABCDEFGHJKLMNPQRSTUVWXYZ0123456789"), "pid": lambda: pick(["SC", "SP", "PC", "RX", "EC", "PF", "DZ", "CN"]) + str(R.randint(10000, 99999)),
        "ord": lambda: str(R.randint(1000, 99999)), "date": lambda: f"{R.randint(1,28)} {pick(['Nov','Dec','Jan','Feb'])}",
        "time": lambda: pick(["9 AM", "10 AM", "11:30 AM", "2 PM", "3 PM", "4 PM", "5 PM", "7 PM"]), "hours": lambda: pick(["1", "2", "6", "12", "24", "48"]),
        "mins": lambda: pick(["5", "10", "15", "30"]), "pct": lambda: pick(["10", "15", "20", "25", "30", "40", "50", "60", "70"]),
        "gb": lambda: pick(["1GB", "2GB", "5GB", "10GB", "15GB", "25GB", "30GB"]), "days": lambda: pick(["3", "7", "15", "30"]),
        "short": lambda: pick(["16247", "16167", "16216", "121", "123", "16268", "09612-XXXXXX", "09666-XXXXXX"]),
        "ussd": lambda: pick(["*247#", "*167#", "*322#", "*121*3#", "*123*1#", "*500#", "*166*6#"]),
        "income": lambda: money(pick([500, 1000, 1500, 2000, 3000, 5000, 10000])), "salary": lambda: money(pick([25000, 30000, 40000, 45000, 60000, 80000])),
        "bnamt": lambda: bn_money(pick([50, 100, 200, 500, 1000, 1500, 2000, 5000, 10000])), "bnbig": lambda: bn_money(pick([10000, 50000, 100000, 500000])),
        "bnmfs": lambda: pick(["বিকাশ", "নগদ", "রকেট", "উপায়", "bKash", "Nagad"]), "bnpct": lambda: pick(["১০", "২০", "২৫", "৩০", "৪০", "৫০"]),
        "bncode": lambda: str(R.randint(100000, 999999)).translate(BN_DIG), "bncourier": lambda: pick(["সুন্দরবন কুরিয়ার", "পাঠাও কুরিয়ার", "রেডএক্স", "স্টেডফাস্ট", "DHL", "FedEx"]),
        "bntelco": lambda: pick(["গ্রামীণফোন", "রবি", "বাংলালিংক", "টেলিটক", "GP", "Robi"]),
    }
    def fill(t):
        return re.sub(r"\{(\w+)\}", lambda m: SLOT[m.group(1)](), t)

    # (template, label, techniques, action)  -- techniques as string, "" = NONE
    T = []
    def add(label, tech, action, *tpls):
        for t in tpls: T.append((t, label, tech, action))

    # ---------------- Class 0: legitimate (many hard negatives) ----------------
    add(0, "", 0,
        "{mfs} OTP: {code}. Do not share it with anyone. Valid for {mins} minutes.",
        "Your {mfs} verification code is {code}. {mfs} will never ask for this code.",
        "{bank} OTP {code} for {amt} payment to {shop}. Never share this OTP with anyone.",
        "{mfs} OTP: {code}. Eta kauke bolben na. Meyad {mins} minute.",
        "{mfs} OTP: {code}. Kauke share korben na. {mins} minute valid.",
        "আপনার {bnmfs} ভেরিফিকেশন কোড {bncode}। কাউকে এই কোড দিবেন না।",
        "Your OTP is {code}. Do not share it with anyone, including our staff.",
        "{mfs}: You have received {amt} from {phone}. Fee Tk 0.00. Balance {amt}. TrxID {trx}",
        "{mfs}: Cash out {amt} successful. Fee {fee}. Balance {amt}. TrxID {trx}",
        "{mfs}: Send money {amt} to {phone} successful. Balance {amt}. TrxID {trx}",
        "{mfs}: Apni {amt} peyechen {phone} theke. Balance {amt}. TrxID {trx}",
        "Apnar cash-out {amt} successful. Balance {amt}.",
        "{mfs}: {shop} e {amt} payment successful. TrxID {trx}.",
        "আপনার {bnamt} ক্যাশ আউট সফল হয়েছে। বর্তমান ব্যালেন্স {bnamt}।",
        "আপনার {bnmfs} অ্যাকাউন্টে {bnamt} জমা হয়েছে।",
        "{bank}: BDT {code4} debited from A/C XXXX{code4} at {shop}. Avl bal BDT {code}.",
        "{bank}: Your A/C XXXX{code4} is credited with {amt} (salary). Avl bal {amt}.",
        "Dhonyobad! Apnar {amt} recharge successful. Balance {amt}.",
        "Your {telco} bill of {amt} is due on {date}. Pay via the {telco} app.",
        "Your {util} bill for this month is {amt}. Due {date}. View/pay: {glink}",
        "প্রিয় গ্রাহক, আপনার {util} বিল {bnamt}। শেষ তারিখ {date}।",
        "{util}: Your payment of {amt} was successful. Receipt no. {pid}.",
        "Reminder: your loan installment {amt} is due tomorrow. Pay via {mfs} Pay Bill or at branch.",
        "Your {shop} order #{ord} has been shipped and will arrive by {date}. Track: {glink}",
        "Your order #{ord} is out for delivery today.",
        "{courier}: Parcel {pid} arrived at the hub. Collect from branch with this SMS and a photo ID.",
        "{courier}: Apnar parcel {pid} ajke deliver hobe. COD {amt} ready rakhun.",
        "{courier}: Your order {pid} was delivered. Thank you for choosing us.",
        "{courier}: Your parcel is waiting. Delivery fee {fee} will be collected in cash by our rider. No online payment is needed.",
        "{shop}: Your rider {name} is arriving in {mins} min. Share your trip PIN only with the rider.",
        "{courier}: আপনার পার্সেল {pid} আজ ডেলিভারি হবে। ধন্যবাদ।",
        "Your {shop} refund of {amt} has been credited to your {mfs} account.",
        "{mfs}: You earned {fee} cashback on your bill payment. It has been credited. Balance {amt}.",
        "{shop}: Your cashback of {fee} for last month's orders has been added to your wallet.",
        "Security alert: new device logged in to your {mfs} account. If this was you, ignore. If not, call {short}. We never ask for your PIN or OTP.",
        "{bank}: Your card ending {code4} was used for {amt} at {shop}. If not you, call {short}.",
        "{mfs} never asks for your PIN or OTP. Beware of fraud calls. Helpline {short}.",
        "Sabdhan! Keu {mfs} PIN ba OTP chaile dibi na. Eta scam.",
        "Hi, this is {name} from the dentist's office. Your appointment is tomorrow at {time}. Reply C to confirm.",
        "Interview schedule: Tuesday {time}, Gulshan office. Bring CV and NID copy. - HR, {company}",
        "Registration for Spring starts {date}. Pay tuition through the student portal before the deadline.",
        "NSU Library: The book you reserved is available. Please collect within 2 days.",
        "Class kal {time} e hobe. Sir mail koreche.",
        "Boss: Meeting moved to {time}. Please join the Google Meet link in the calendar invite.",
        "Mama, ajke raate bari asbo. Bazar theke kichu lagbe?",
        "Bhai kal exam er routine ta pathaiso?",
        "Ami ekhon {mfs} e {amt} pathaisi, dekh to ashche kina.",
        "Ammu, {amt} pathalam {mfs} e, check koro.",
        "Lunch er bill {amt}, sobai {fee} kore pathao.",
        "Happy birthday {name}! Have a great year ahead.",
        "Ajke ki office e ashba? Traffic onek.",
        "Welcome to the study group. Rules: no spam, no links.",
        "Your {telco} {gb} pack expires on {date}. Dial {ussd} to renew.",
        "Dear customer, your {telco} internet pack has been activated successfully.",
        "Your {shop} account password was changed. If this wasn't you, reset it from the app.",
        "Western Union: {amt} remittance has been credited to your {bank} account. Ref {trx}.",
        "kal dekha hobe inshallah",
        "ok thik ache, ami 5 tay ashtesi",
        "আগামীকাল ক্লাস সকাল ১০টায় হবে।",
        "আম্মু, আমি ঠিকমতো পৌঁছেছি।",
    )
    # ---------------- Class 1: MFS / OTP / credential scam ----------------
    add(1, "T1|T2|T5", 1,
        "{mfs} Helpline: Apnar account e unusual login detect hoyeche. Block theke bachte 6 digit code ta ei number e reply korun.",
        "Dear {mfs} user, KYC update na korle account {hours} ghontar moddhe band hobe. Update korte NID number o PIN pathan.",
        "Your {mfs} PIN will expire today. Reply with your current PIN and a new PIN to continue service.",
        "{mfs} Alert: your account will be blocked within {hours} hours. Confirm your password and OTP here: {link}",
        "Notice: your account will be blocked within {hours} hours. Tell us your card number, expiry and CVV.",
        "{bank}: Dear user, we detected suspicious activity on your account. Share the OTP you just received to stop the transaction.",
        "{mfs} customer care: apnar account {hours} ghontar moddhe band hoye jabe. je 6 digit code ashche seta bolen.",
        "{mfs} theke bolchi. apnar wallet limit barano hobe. code ta {phone} e pathan.",
        "{bank} security department theke apnar wallet limit barano hobe, PIN ta ekhoni reply korun.",
        "Dear user, your card ending {code4} has been flagged. Verify your card number and CVV at {link} to avoid suspension.",
        "{mfs}: vul transaction e {amt} kata hoyeche. NID number ar jonmo tarikh pathan, na hole account suspend hobe.",
        "প্রিয় গ্রাহক, আপনার {bnmfs} অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান।",
        "আপনার {bnmfs} অ্যাকাউন্ট যাচাই করতে ওটিপি কোডটি এই নম্বরে পাঠান। না হলে অ্যাকাউন্ট বন্ধ হবে।",
        "Apnar SIM verify korte NID er front back photo ar OTP pathan, na hole SIM off hoye jabe. - {telco}",
        "{gov}: Your NID needs re-verification. Send your NID number, date of birth and the OTP you receive.",
        "Dear user, we are verifying a payment of {amt} to {shop}. Confirm your password and OTP here: {link}",
    )
    add(1, "T1|T2", 1,
        "{shop} refund team: send card number, expiry and CVV to receive your {amt} refund.",
        "Western Union: {amt} remittance from your brother in {country} is ready. Verify by sending NID photo and wallet PIN.",
        "{courier}: Apnar parcel {pid} atke ase. Release korte PIN ar OTP den.",
        "{courier}: {fname} from {country} sent you a gift parcel. Enter your card number and CVV at {link} to receive it.",
        "{shop}: Your order #{ord} needs verification. Send the OTP you received to confirm delivery.",
        "{gov}: Apnar stipend er taka pathano hobe. {mfs} number ar PIN pathan.",
    )
    add(1, "T1|T4|T2", 1,
        "{mfs} bonus: apni {amt} bonus peyechen. Wallet e add korte {ussd} dial kore PIN din.",
        "{mfs} Eid Bonus: {amt} is yours! Send your PIN to receive.",
        "Congratulations! You won a {big} prize in the {mfs} draw. Enter your card number and CVV at {link}",
        "Congratulations! Apni {big} jitechen. Claim korte apnar {mfs} OTP ta share korun.",
        "অভিনন্দন! আপনি {bnamt} জিতেছেন। ক্লেইম করতে ওটিপি দিন।",
    )
    add(1, "T2|T5", 1,
        "Ami agent, bhul number-e taka pathaisi. Ferot din, PIN ta bolen.",
        "Mama ami bipode porechi, {mfs} e taka nai. Tomar PIN ta dao ami ektu send kore nei.",
        "Bhai ekta code gese tomar phone e bhul kore, oi code ta ektu bolo plz, urgent.",
        "Apu, amar number e ekta OTP ashar kotha chilo, tomar number e chole gese. code ta pathao please.",
    )
    add(1, "T3|T5", 2,
        "Bhai ekta bhul hoye geche. Amar {amt} vul kore apnar number e chole gese. Please ferot den {phone} e. Ami onek bipode.",
        "bhai bhul kore tomar number e {amt} recharge diye felsi, ferot dao pls {phone} e flexiload kore",
        "Sorry, I sent {amt} to your {mfs} by mistake. Please send it back to {phone}, it is my hospital money.",
        "ভুল করে আপনার নম্বরে {bnamt} চলে গেছে। দয়া করে টাকাটা ফেরত দিন।",
    )
    # ---------------- Class 2: prize / lottery ----------------
    add(2, "T4|T5", 3,
        "You won cashback. Click to claim within {hours} hour.",
        "Apni {big} jitechen. Claim korte link e click korun. {link}",
        "অভিনন্দন! আপনি {bnbig} ক্যাশব্যাক জিতেছেন। পেতে নিচের লিংকে ক্লিক করুন।",
        "Congratulations! Your number won {big} in the monthly lucky draw. Claim now: {link} Offer valid {hours} hours only.",
        "YOU HAVE BEEN SELECTED FOR A {big} PRIZE! CLICK {link} BEFORE MIDNIGHT.",
    )
    add(2, "T1|T4|T5", 3,
        "Free {gb} data gift from {telco} for you! Tap {link} now, offer expires in {hours} hour.",
        "{mfs} anniversary: claim your {amt} gift within {hours} hours: {link}",
        "FREE iPhone for {telco} customers! Take the 2-min survey: {link}",
    )
    add(2, "T1|T4", 3,
        "{shop} anniversary gift: answer 3 questions and win a {amt} voucher {link}",
        "{brand} customer reward: you have been chosen for a free gift hamper. Details: {link}",
    )
    add(2, "T1|T4", 4,
        "{mfs} anniversary: spin and win {amt}! Download the spin app: {link}",
        "{shop} scratch card: install our app to reveal your prize {link}",
    )
    add(2, "T1|T3|T4", 2,
        "You won cashback {amt} from {telco} lucky draw! Pay {fee} processing fee to claim.",
        "{mfs} Lucky Draw: Apnar {mfs} account e {big} reward ashche. Delivery charge {fee} {mfs} korun.",
        "{mfs} ঈদ বোনাস! {bnamt} পেতে প্রসেসিং ফি {bnamt} দিন।",
        "{gov}: Your child is selected for a {amt} stipend. Pay a {fee} verification fee to receive it.",
    )
    add(2, "T4|T3", 2,
        "Dear winner, you won cashback of {amt}! Pay {fee} processing fee to receive it.",
        "Lucky draw e apni ekta motorbike jitechen! Registration fee {fee} pathan {phone} e.",
    )
    add(2, "T1|T4|T5", 5,
        "Congratulations! Apnar number ti {telco} Eid lucky draw te select hoyeche. iPhone jitar jonno {phone} e WhatsApp korun.",
        "{mfs} Eid Bonus: Apni {amt} cashback jitechen. {phone} e WhatsApp korun. 1 ghontar moddhe claim korun.",
    )
    add(2, "T4", 1,
        "Lucky draw winner! Reply with your name and address to receive a Toyota car.",
        "Luckydraw winner! NID photo pathan prize nite.",
    )
    # ---------------- Class 3: fake job / investment ----------------
    add(3, "T6", 4,
        "Task: buy products on {shop} via our link, we refund + 20% commission. Complete 5 tasks, get {income}.",
        "Part-time job: like YouTube videos, earn {income} per task. Join Telegram: {link}",
        "Ghor bose din e {income} income. WhatsApp e join korun.",
        "ঘরে বসে প্রতিদিন {bnamt} আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।",
        "Facebook page like korle protek like e 5 taka. Group e join korun.",
        "Mobile diye online income, daily {income}. Amader Telegram group e join korun: {link}",
        "Rate 5-star reviews on Google Maps and earn {income} daily. Install the app: {link}",
    )
    add(3, "T6|T5", 4,
        "Crypto signals group - 300% weekly profit guaranteed. Join VIP: {link}",
        "Only 10 seats left! Earn {income} daily from home. Join our WhatsApp group now.",
    )
    add(3, "T1|T3|T6", 2,
        "Congratulations, you are selected for Data Entry Officer at {company}. Pay {fee} registration fee to confirm.",
        "{company} is hiring! Salary {salary}/month, work from home. Pay {fee} registration fee to start. Limited seats!",
        "Selected as brand ambassador for {shop}! Earn {income} per post. Pay {fee} for your kit.",
        "Selected for data entry at {company}. Pay {fee} registration fee to start. Apply today, offer closes tonight.",
        "S: Hi, I saw your CV on Bdjobs. We have a work-from-home position ||| R: what is the salary? ||| S: {salary} per month ||| R: ok ||| S: You need to deposit {fee} as security first.",
    )
    add(3, "T3|T6", 2,
        "Double your money in 7 days. Invest {amt}, get double. {mfs} {phone}",
        "Invest {amt} and get double in {days} days, 100% guaranteed profit.",
        "Crypto bot trading, 30% daily profit guaranteed. Pay {fee} registration fee to start.",
        "Freelancing course: earn $500/month from home guaranteed. Enroll now with {fee}.",
        "Deposit {amt} to unlock tasks and earn 3x commission every day.",
        "টাকা দ্বিগুণ করার সুযোগ! {bnamt} বিনিয়োগ করুন, ৭ দিনে দ্বিগুণ পান।",
        "Forex trading e invest korun, mashe 50% profit nischit. Minimum {amt}.",
    )
    add(3, "T3|T6", 5,
        "Abroad job in {country}, visa fee {amt}. 100% guaranteed. Call {phone}",
        "{country} e factory job, salary {salary}. Processing fee lagbe. Details janate call korun {phone}",
    )
    # ---------------- Class 4: delivery / parcel phishing ----------------
    add(4, "T1|T3", 2,
        "{courier}: Parcel {pid} held at the hub. Pay {fee} storage fee: {link}",
        "Customs hold: your parcel from {country} needs {amt} duty. Pay via {mfs} to release.",
        "{courier}: Customs duty is pending for your package. Pay {fee} via {mfs} {phone}.",
        "{courier}: {country} theke ekta parcel apnar jonno ashche. Release korte {fee} fee din: {link}",
        "{courier}: Parcel {pid} er address adhura. Release korte {fee} fee din: {link} Taratari korun.",
        "{courier}: a gift parcel from {fname} ({country}) is waiting for you. Pay {fee} fee to proceed.",
        "{courier}: {fname} apnake gift parcel pathiyeche, customs e atke ase. Release korte {fee} pathan {phone} e.",
    )
    add(4, "T3", 2,
        "Parcel atke ase. Delivery fee dite link e jan.",
        "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ {bnamt} পরিশোধ করতে লিংকে ক্লিক করুন।",
        "Parcel {pid} deliver kora jayni. Release korte {fee} fee din: {link}",
        "Your package is on hold due to unpaid shipping fee of {fee}. Pay at {link}",
    )
    add(4, "T1|T5", 3,
        "Courier notice: package #{pid} requires address confirmation. Click {link} within {hours}h.",
        "{courier}: Your parcel could not be delivered. Confirm address at {link}",
        "{courier}: ডেলিভারি ব্যর্থ হয়েছে। ঠিকানা আপডেট করতে {link} এ যান।",
    )
    add(4, "T5", 3,
        "Delivery attempt failed. Update address within {hours} hours or parcel will be returned. {link}",
        "Your package could not be delivered. Enter your address details at {link}",
    )
    add(4, "T1|T5", 5,
        "{courier}: Apnar parcel er address vul. Call korun {phone} ekhoni.",
        "{courier}: Your parcel {pid} is on hold. Call {phone} within {hours} hours to avoid return.",
    )
    # ---------------- Class 5: promotional spam ----------------
    add(5, "", 0,
        "Get {pct}% cashback on your next top-up with code EID{pct}.",
        "Eid offer! Sob item e {pct}% discount.",
        "{shop} 11.11 sale starts tonight 12 AM. Vouchers up to {amt}.",
        "{shop}: Flash sale ends tonight! Extra {pct}% off. Use code DEAL{pct}. T&C apply.",
        "{telco}: Winter sale is live! Up to {pct}% off.",
        "{telco} Mega Offer! Buy {gb} get {gb} free. Dial {ussd}",
        "{telco}: Special offer! {gb} for {fee}, {days} days. Dial {ussd}",
        "Learn English in 30 days! Admission going on. Call now.",
        "{brand}: Admission going on for Digital Marketing course. Details: {glink}",
        "Install our app and get {fee} off your first order. Code: WELCOME{pct}",
        "{shop}: Recharge {fee} and get {gb} extra. Open the app to apply.",
        "Pizza Hut combo: 2 large pizzas + drink for {amt} today only!",
        "Book your dream apartment in Bashundhara, easy installment. Call {short}",
        "{brand}: Book your dream flat, easy installment. Details: {link} T&C apply.",
        "{shop}: End of season sale up to {pct}% off. Visit your nearest outlet. T&C apply.",
        "{bank}: 0% EMI for 12 months on all purchases at {shop}.",
        "{mfs}: Bill payment e {pct}% cashback. Dial {ussd}. Shortto projojjo.",
        "{mfs}: Apnar jonno special offer: {pct}% off. {short} e call korun. Shortto projojjo.",
        "{telco}: Eid offer! Sob item e {pct}% discount. {short} e call korun. Limited time!",
        "Refer a friend and earn {fee} credit on your next order. Dial {ussd}. Limited time only.",
        "Order 3 times this week and unlock a {fee} voucher. Offer ends Sunday.",
        "{brand}: New app update with exciting features. Call {short}. Limited time only.",
        "{shop}: weekend sale shuru, {pct}% porjonto chhar. {short} e call korun.",
        "ঈদ উপলক্ষে সব পণ্যে {bnpct}% ছাড়! আজই কিনুন।",
        "{bntelco}: মাত্র {bnamt} টাকায় ৩০ জিবি ইন্টারনেট। আজই কিনুন।",
        "Sir, hair fall problem? Our herbal oil is very popular. 40% discount today only, order now.",
        "{fee} recharge e {gb} extra pan. Nikotbortee showroom e ashun.",
        "Winter collection is here! Visit {shop} outlets or shop online at {glink}",
        "{bank} credit card: Buy 1 Get 1 buffet at partner hotels this weekend.",
    )

    # ---------------- extra Bangla-script templates (all classes) ----------------
    add(0, "", 0,
        "{mfs}: আপনি {bnamt} পেয়েছেন। ব্যালেন্স {bnamt}। TrxID {trx}",
        "আপনার {bnamt} রিচার্জ সফল হয়েছে। ধন্যবাদ।",
        "{bnmfs} ওটিপি {bncode}। এই কোড কাউকে দেবেন না। মেয়াদ ৫ মিনিট।",
        "আপনার অর্ডার #{ord} শিপ করা হয়েছে, {date} এর মধ্যে পৌঁছাবে।",
        "{bncourier}: আপনার পার্সেল {pid} হাবে পৌঁছেছে। এনআইডি দেখিয়ে শাখা থেকে সংগ্রহ করুন।",
        "সতর্কতা: {bnmfs} কখনো আপনার পিন বা ওটিপি চায় না। কাউকে শেয়ার করবেন না।",
        "আপনার {bnamt} রিফান্ড আপনার {bnmfs} অ্যাকাউন্টে জমা হয়েছে।",
        "ভাই, কাল পরীক্ষার রুটিনটা পাঠাবি?",
        "আজ রাতে বাসায় আসবো, কিছু লাগবে?",
        "আপনার বিদ্যুৎ বিল {bnamt}, পরিশোধের শেষ তারিখ {date}।",
        "মিটিং বিকেল ৩টায় পিছানো হয়েছে।",
    )
    add(1, "T1|T2|T5", 1,
        "{bnmfs} থেকে বলছি। আপনার অ্যাকাউন্ট ২৪ ঘণ্টার মধ্যে বন্ধ হবে। আপনার ফোনে আসা ৬ সংখ্যার কোডটি বলুন।",
        "প্রিয় গ্রাহক, কেওয়াইসি আপডেট না করলে আপনার {bnmfs} অ্যাকাউন্ট বন্ধ হবে। এনআইডি নম্বর ও পিন পাঠান।",
        "আপনার কার্ড ব্লক হয়ে গেছে। চালু করতে কার্ড নম্বর ও সিভিভি পাঠান।",
        "আপনার সিম বন্ধ হয়ে যাবে। যাচাই করতে এনআইডির ছবি ও ওটিপি পাঠান।",
    )
    add(1, "T2|T5", 1, "ভাই, ভুল করে আপনার নম্বরে একটা কোড গেছে, কোডটা একটু বলবেন? খুব জরুরি।")
    add(1, "T1|T4|T2", 1, "{bnmfs} বোনাস: আপনি {bnamt} পেয়েছেন। পেতে আপনার পিন নম্বর পাঠান।")
    add(2, "T4|T5", 3,
        "অভিনন্দন! আপনার নম্বর লটারিতে {bnbig} জিতেছে। এখনই লিংকে ক্লিক করুন: {link}",
        "আপনি একটি ফ্রি আইফোন জিতেছেন! ১ ঘণ্টার মধ্যে ক্লেইম করুন: {link}")
    add(2, "T1|T3|T4", 2,
        "{bntelco} লাকি ড্র: আপনি {bnbig} জিতেছেন। প্রসেসিং ফি {bnamt} পাঠান।",
        "অভিনন্দন! আপনি মোটরসাইকেল জিতেছেন। রেজিস্ট্রেশন ফি {bnamt} বিকাশ করুন।")
    add(2, "T1|T4", 5, "{bntelco} ঈদ লাকি ড্র তে আপনার নম্বর নির্বাচিত হয়েছে। পুরস্কার পেতে {phone} নম্বরে হোয়াটসঅ্যাপ করুন।")
    add(3, "T6", 4,
        "মোবাইল দিয়ে অনলাইনে আয় করুন, দৈনিক {bnamt}। টেলিগ্রাম গ্রুপে যোগ দিন: {link}",
        "ফেসবুক পেজে লাইক দিয়ে প্রতি লাইকে ৫ টাকা আয় করুন। গ্রুপে জয়েন করুন।")
    add(3, "T1|T3|T6", 2,
        "অভিনন্দন! আপনি {company} এ ডাটা এন্ট্রি পদে নির্বাচিত হয়েছেন। রেজিস্ট্রেশন ফি {bnamt} পাঠান।",
        "ঘরে বসে চাকরি, বেতন মাসে ৪০,০০০ টাকা। জামানত হিসেবে {bnamt} জমা দিন।")
    add(3, "T3|T6", 2, "ক্রিপ্টোতে বিনিয়োগ করুন, প্রতিদিন ১০% লাভ নিশ্চিত। সর্বনিম্ন {bnamt}।")
    add(4, "T1|T3", 2,
        "{bncourier}: বিদেশ থেকে আপনার একটি পার্সেল এসেছে। কাস্টমস ফি {bnamt} পরিশোধ করুন: {link}",
        "{bncourier}: আপনার পার্সেল {pid} আটকে আছে। ছাড়াতে {bnamt} ফি দিন।")
    add(4, "T1|T5", 3, "{bncourier}: আপনার ঠিকানা অসম্পূর্ণ। ২৪ ঘণ্টার মধ্যে এই লিংকে ঠিকানা নিশ্চিত করুন: {link}")
    add(4, "T1|T5", 5, "{bncourier}: আপনার পার্সেলের ঠিকানা ভুল। এখনই {phone} নম্বরে কল করুন।")
    add(5, "", 0,
        "{bntelco}: মাত্র {bnamt} টাকায় {gb} ইন্টারনেট, মেয়াদ ৭ দিন। ডায়াল করুন {ussd}",
        "শীতকালীন সেল! সব পোশাকে {bnpct}% পর্যন্ত ছাড়। আজই আউটলেটে আসুন।",
        "{bnmfs}: বিল পেমেন্টে {bnpct}% ক্যাশব্যাক। অফারটি সীমিত সময়ের জন্য।",
        "ইংরেজি শিখুন মাত্র ৩০ দিনে! ভর্তি চলছে। আজই কল করুন।",
        "নতুন অ্যাপ ডাউনলোড করে প্রথম অর্ডারে {bnamt} ছাড় পান। কোড: NEW50",
    )

    # ---------------- v3: extra templates for error types seen on sample.csv (no test text used) ----------------
    # legitimate notices that look like telecom / courier / job / institution messages
    add(0, "", 0,
        "Dear customer, your {telco} {gb} pack will expire on {date}. To renew, dial {ussd} or use the {telco} app.",
        "{telco}: Your remaining data is 200MB. Your pack is valid till {date}. Thank you for staying with {telco}.",
        "{telco}: Your SIM has been re-registered successfully with your NID. No further action is needed.",
        "Priyo grahok, apnar {telco} internet pack {date} e sesh hobe. Renew korte {ussd} dial korun.",
        "{telco}: Apnar minute pack er meyad ajke rat 12 tay sesh hobe.",
        "{courier}: Parcel {pid} has reached our {name} Road branch. Please collect it within 3 days with your NID.",
        "{courier}: Your parcel {pid} could not be delivered today because nobody was home. We will try again tomorrow.",
        "{courier}: Apnar parcel {pid} branch e pouche geche. Office time e eshe niye jan.",
        "{courier} rider {name}: bhai ami apnar basar niche, parcel ta niye jan please.",
        "Your interview at {company} is confirmed for {date}, {time}. Please bring your CV and certificates. Reception will guide you.",
        "{company} HR: Thank you for applying. Your written test is on {date} at {time} at our head office. No fee is required.",
        "HR, {company}: Your offer letter has been sent to your email. Please sign and return it by {date}.",
        "Interview kal {time} e, {company} office e. CV er hard copy niye ashben.",
        "University notice: Mid-term exams start on {date}. Check the routine on the student portal.",
        "Library notice: Please return the borrowed book by {date} to avoid a late fine.",
        "Dear student, your tuition payment of {amt} has been received. Receipt no. {pid}.",
        "Hospital: Your appointment with Dr. {name} is on {date} at {time}. Please arrive 15 minutes early.",
        "Team, the meeting is moved to {time}. Same Zoom link as last week.",
        "Bhai kal class {time} theke shift kore {time} e hobe, sir group e janaise.",
        "Amar train late, {time} er dike pouchabo. Tumi kheye nio.",
        "Doctor er serial dite hobe kal, tumi ektu call kore dite parba?",
        "R: I forgot my {mfs} PIN, what should I do? ||| S: Dial {ussd} and follow the reset steps, or visit a customer care centre with your NID. Never share your PIN with anyone.",
        "R: amar {mfs} account lock hoye geche ||| S: {short} e call korun ba nearest service point e NID niye jan. Kauke PIN ba OTP diben na.",
        "{mfs}: Your PIN was changed successfully. If you did not do this, call {short} immediately.",
        "{bank}: Your new debit card is ready for collection at the branch. Please bring your NID.",
        "{bank}: Your monthly e-statement for A/C XXXX{code4} has been sent to your email.",
        "{gov}: Your application {pid} has been received. You will be notified by SMS about the next step.",
        "আপনার {bntelco} ইন্টারনেট প্যাকের মেয়াদ আগামীকাল শেষ হবে। নবায়ন করতে অ্যাপ ব্যবহার করুন।",
        "{bncourier}: আপনার পার্সেল শাখায় পৌঁছেছে। জাতীয় পরিচয়পত্র নিয়ে সংগ্রহ করুন।",
        "আগামীকাল সকাল ১০টায় ইন্টারভিউ, সিভি ও সার্টিফিকেট সাথে আনবেন।",
    )
    # promotions that are not scams (real-estate, installments, services)
    add(5, "", 0,
        "Book your flat in {name} Tower, Bashundhara. Easy installment, handover in 2027. Call {short}.",
        "Ready flat in Uttara! 1350 sqft, easy EMI. Visit our project this Friday.",
        "{bank} credit card: 0% EMI on electronics at {shop} this month. T&C apply.",
        "{shop}: New arrivals in store! Visit your nearest outlet this weekend.",
        "Admission open for HSC batch at our coaching centre. Free trial class on {date}.",
        "Gym membership {pct}% off this month only. Visit us today.",
        "{telco}: Get {gb} bonus on every recharge of {fee} this week. Dial {ussd}",
        "Flat booking cholche! Sohoj kistite nijer flat. Details er jonno call korun {short}.",
    )
    # credential request inside a prize message -> class 1 (credential requests take priority)
    add(1, "T1|T2|T4|T5", 1,
        "Congratulations! Apnar number {mfs} lucky draw e {big} jitese. Taka pete OTP ta amader bolun.",
        "You won {big} in the {telco} Eid draw! Reply with the OTP we just sent to claim it.",
        "Abhinandan! Apni {amt} cashback peyechen. Pete {mfs} PIN ar OTP pathan {phone} e.",
        "অভিনন্দন! আপনি {bnamt} পুরস্কার জিতেছেন। টাকা পেতে আপনার পিন ও ওটিপি পাঠান।",
    )
    # task-based earning that looks like shopping promotions -> class 3
    add(3, "T3|T6", 4,
        "Shop on {shop} with our link and get a full refund plus {pct}% commission per order. Finish 3 tasks a day.",
        "Order task: buy any item on {shop} through our app, we refund the price and pay {income} commission.",
        "Product review job: buy, review and get the money back with {pct}% extra. Join our WhatsApp group.",
        "{shop} er product order kore review din, taka ferot + {pct}% commission. Telegram e join korun: {link}",
    )

    # ---------------- composition helpers ----------------
    by_label = {k: [t for t in T if t[1] == k] for k in range(6)}
    MARK = [("S", "R"), ("A", "B"), ("[Sender]", "[You]"), ("Caller", "You"), ("s", "r"), ("a", "b"), ("[sender]", "[you]"), ("Agent", "Me"), ("", "")]
    OPEN = [("hello", "hi, who is this?"), ("assalamu alaikum", "walaikum assalam, ke bolchen?"), ("Hi, is this {name}?", "yes, speaking"),
            ("brother, ekta help lagbe", "ki help?"), ("Good evening sir", "yes?"), ("apni ki {name}?", "ji bolen"), ("Hello", "ke?")]
    REPLY = ["ok", "I see", "sure", "ki?", "kothay?", "accha", "what?", "hmm", "really?", "ok bolen", "ami bujhlam na"]
    WRAP_PRE = ["We hope this message finds you well. ", "This is an automated notification from our system. ", "Dear customer, ",
                "As part of our regular communication, please note: ", "Notice: ", "Subject: Important update\n", "⚠️ ", "Sabdhan! ", "FYI: "]
    WRAP_SUF = [" Thank you for your cooperation.", " Please do not reply to this automated message.", " - Customer Care", " Dhonnobad.", " 😊", " 🎁", " 📲"]
    WARN = ["eta scam bhai, kauke PIN ba OTP dibi na, block kor", "this is a scam, don't reply and report the number", "eta fake, ignore kor",
            "don't click that link, it's phishing", "eta fraud, {mfs} kokhono PIN chay na"]

    def render(k):
        t, lab, tech, act = pick(by_label[k])
        return fill(t), lab, tech, act, T.index((t, lab, tech, act))

    def as_conversation(text, lab):
        a, b = pick(MARK)
        sp = (lambda s, x: f"{s}: {x}" if s else x)
        turns = []
        if R.random() < 0.6:
            for _ in range(R.randint(1, 3)):
                o1, o2 = pick(OPEN)
                turns += [sp(a, fill(o1)), sp(b, fill(o2))]
        parts = [p.strip() for p in re.split(r"(?<=[.!?।])\s+", text) if p.strip()]
        if len(parts) >= 2 and R.random() < 0.7:
            cut = R.randint(1, len(parts) - 1)
            turns += [sp(a, " ".join(parts[:cut])), sp(b, pick(REPLY)), sp(a, " ".join(parts[cut:]))]
        else:
            turns.append(sp(a, text))
            if R.random() < 0.4: turns.append(sp(b, pick(REPLY)))
        return " ||| ".join(turns[-12:])

    def group_chat(text):
        n1, n2 = R.sample(NAME, 2)
        chat = [f"[Group] {n1}: {pick(['anyone has the slides?', 'thanks bro', 'kal class hobe?', 'ok'])}",
                f"[Group] {n2}: {pick(['i will share tonight', 'ha hobe', 'np'])}"]
        return " ||| ".join(chat + [f"[Group] {phone()}: {text}"])

    rows = []
    mix = [0] * 30 + [1] * 15 + [2] * 12 + [3] * 13 + [4] * 13 + [5] * 17
    while len(rows) < n_rows:
        k = pick(mix)
        text, lab, tech, act, tid = render(k)
        u = R.random()
        if k == 0 and u < 0.12:     # warning conversation quoting a scam -> legitimate
            s_text, *_, tid = render(pick([1, 2, 3, 4]))
            a, b = pick(MARK[:4])
            text = f"{a}: Someone from '{pick(MFS + COURIER)} helpline' messaged me: '{s_text}' ||| {b}: {fill(pick(WARN))}"
            if R.random() < 0.5: text += f" ||| {a}: {pick(['should I report it?', 'ok thanks', 'thik ache'])}"
        elif k == 5 and u < 0.12:   # two promos in a row
            text = text + " ||| " + render(5)[0]
        elif u < 0.35:
            text = as_conversation(text, lab)
        elif k in (1, 2, 3, 4) and u < 0.42:
            text = group_chat(text)
        if R.random() < 0.25: text = pick(WRAP_PRE) + text
        if R.random() < 0.2: text = text + pick(WRAP_SUF)
        r = R.random()
        if r < 0.15: text = text.lower()
        elif r < 0.18: text = text.upper()
        rows.append((text, lab, tech or "NONE", act, tid))
    # noise-only legit rows
    for _ in range(max(1, n_rows // 200)):
        rows.append((" ".join(rnd_id(R.randint(5, 11), "qwrtypsdfghjklzxcvbnm") for _ in range(R.randint(1, 2))), 0, "NONE", 0, -1))
    df = pd.DataFrame(rows, columns=["text", "label", "techniques", "action", "tid"])
    canon = [fill(t[0]) for t in T]          # one rendering per template, used to detect templates similar to real rows
    return df, canon

# ---------------- realistic noise augmentation ----------------
HOMO = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "y": "у", "x": "х", "i": "і"}
LEET = {"a": "4", "e": "3", "o": "0", "i": "1", "s": "5", "t": "7"}
MASH = ["wkrjfejejrn", "zzqwpo", "mnbvcx", "qzxvbn", "xkcdpl", "asdkjh", "lkjsdfoiu", "plmokn", "trvbgh"]
EMO = ["🎁", "💰", "🔥", "⚠️", "😊", "📲", "🙏", "✅"]
def corrupt(text, rng, strength=1.0):
    chars = list(text); out = []
    p = 0.04 * strength
    for ch in chars:
        r = rng.random()
        if ch.isalpha() and r < p:
            op = rng.randint(0, 6)
            if op == 0: continue                                   # deletion
            if op == 1: out += [ch] * rng.randint(2, 5); continue  # repetition
            if op == 2: out += [ch, rng.choice("abcdefghijklmnoprstuvwy")]; continue  # insertion
            if op == 3 and ch.lower() in HOMO: out.append(HOMO[ch.lower()]); continue
            if op == 4 and ch.lower() in LEET: out.append(LEET[ch.lower()]); continue
            if op == 5: out += [ch, "​"]; continue           # zero-width
            out.append(ch.swapcase()); continue
        if ch == " " and r < 2 * p: continue                       # missing space
        out.append(ch)
    s = "".join(out)
    if rng.random() < 0.3 * strength:
        w = s.split(" "); w.insert(rng.randint(0, len(w)), rng.choice(MASH)); s = " ".join(w)
    if rng.random() < 0.25 * strength:
        w = s.split(" "); w.insert(rng.randint(0, len(w)), rng.choice(EMO)); s = " ".join(w)
    if rng.random() < 0.1 * strength: s = s.replace("[.]", "[]")
    return s

synth, TEMPLATE_CANON = make_synth(N_SYNTH, SEED)
n_templates = len(TEMPLATE_CANON)
_rng = random.Random(SEED + 1)
noisy_mask = np.array([_rng.random() < NOISE_FRAC for _ in range(len(synth))])
synth.loc[noisy_mask, "text"] = [corrupt(t, _rng) for t in synth.loc[noisy_mask, "text"]]
print(f"synthetic rows: {len(synth)} from {n_templates} templates | noised: {noisy_mask.mean():.2f}")
print("label mix:", synth.label.value_counts(normalize=True).sort_index().round(3).to_dict())
print("action mix:", synth.action.value_counts().sort_index().to_dict())
for t in synth.sample(8, random_state=SEED).itertuples(): print(f"  [{t.label}|{t.techniques}|{t.action}] {t.text[:150]}")

## 3. Metrics, decoding, validation folds, and Model 1 (TF-IDF + logistic regression)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, RepeatedStratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from scipy.sparse import hstack
from scipy.sparse.csgraph import connected_components

synth["n"] = synth["text"].map(norm)
synth["y"] = synth["label"].astype(int); synth["act"] = synth["action"].astype(int)
TB_real = np.array([tech_to_vec(s) for s in train["techniques"]])

# ---- decoding: probabilities -> (label, action, techniques) with labeling-guide consistency ----
def decode(PL, PA, PT, thr=0.5):
    lab = PL.argmax(1)
    fraud = ~np.isin(lab, [0, 5])
    act = np.where(fraud, PA[:, 1:].argmax(1) + 1, 0)
    tb = (PT >= thr).astype(int)
    none_on = fraud & (tb.sum(1) == 0)
    tb[none_on, PT[none_on].argmax(1)] = 1          # a fraud row needs at least one technique
    tb[~fraud] = 0
    return lab, act, tb

def scores(y, act_true, tb_true, PL, PA, PT, thr=0.5, PL_noisy=None, temp=1.0):
    lab, act, tb = decode(PL, PA, PT, thr)
    FA = f1_score(y, lab, average="macro", labels=range(6), zero_division=0)
    FB = f1_score(tb_true, tb, average="macro", zero_division=0)
    FC = f1_score(act_true, act, average="macro", labels=range(6), zero_division=0)
    conf = softmax_t(PL, temp).max(1)
    CAL = 1 - np.mean((conf - (lab == y)) ** 2)
    FE = f1_score(y, PL_noisy.argmax(1), average="macro", labels=range(6), zero_division=0) if PL_noisy is not None else FA
    S = 100 * (0.40 * FA + 0.15 * FB + 0.10 * FC + 0.10 * FA + 0.10 * FE + 0.05 * CAL + 0.10 * FA)
    return {k: round(float(v), 4) for k, v in dict(S_proxy=S, FA=FA, FB=FB, FC=FC, FA_noisy=FE, CAL=CAL, acc=(lab == y).mean()).items()}

def softmax_t(P, t):
    L = np.log(np.clip(P, 1e-9, 1)) / t
    L -= L.max(1, keepdims=True); E = np.exp(L)
    return E / E.sum(1, keepdims=True)

# ---- team-shared CV folds (Rabib's exact line): 5 folds x 5 repeats = 25 splits ----
splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42).split(np.zeros(len(train)), train['label'].values))
FOLDS = splits[:5]          # first repeat; used where 25 fits would be too slow (XLM-R)
print("real rows", len(train), "| splits", len(splits), "| val sizes (repeat 0)", [len(v) for _, v in FOLDS])

# Template-disjoint validation: drop synthetic templates that resemble a validation row (char cosine > SIM_THR).
# Several templates were written after studying sample.csv, so without this the CV score would leak.
SIM_THR = 0.35
_tv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(list(train["n"]) + [norm(c) for c in TEMPLATE_CANON])
_sim = (_tv.transform([norm(c) for c in TEMPLATE_CANON]) @ _tv.transform(train["n"]).T).toarray()   # templates x real rows
TEMPLATE_NEAR_REAL = _sim.max(1) > SIM_THR
print(f"templates resembling some real row: {TEMPLATE_NEAR_REAL.sum()} / {len(TEMPLATE_CANON)}")
def blocked_templates(val_idx):
    return set(np.where(_sim[:, val_idx].max(1) > SIM_THR)[0])

# corrupted copy of the real rows, used as the robustness validation set
_rng_val = random.Random(SEED + 7)
train["n_noisy"] = [norm(corrupt(t, _rng_val, strength=1.5)) for t in train["text"]]

def build_train(real_idx, extra=None, block=None):
    """synthetic rows (minus blocked templates) + REAL_UP clean copies of the selected real rows.
    Ablation on the shared 25 splits (TF-IDF): noising the real copies lowered FA 0.877 -> 0.865, so they stay clean;
    noise on the synthetic rows is kept (no noise at all: FA 0.852 / noisy 0.804 vs 0.877 / 0.842 with it)."""
    rng = random.Random(SEED + 3)
    s = synth if not block else synth[~synth["tid"].isin(block)]
    parts = [s[["n", "y", "act", "techniques"]]]
    r = train.iloc[real_idx]
    for k in range(REAL_UP):
        c = r[["text", "y", "act", "techniques"]].copy()
        c["n"] = [norm(corrupt(t, rng) if k > 0 and rng.random() < REAL_NOISE_P else t) for t in c["text"]]
        parts.append(c[["n", "y", "act", "techniques"]])
    if extra is not None: parts.append(extra[["n", "y", "act", "techniques"]])
    d = pd.concat(parts, ignore_index=True)
    return d, np.array([tech_to_vec(s) for s in d["techniques"]])

class TfidfModel:
    def fit(self, d, TBd):
        self.cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=300000, dtype=np.float32)
        self.wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, min_df=2, token_pattern=r"\S+", dtype=np.float32)
        X = hstack([self.cv.fit_transform(d["n"]), self.wv.fit_transform(d["n"])]).tocsr()
        mk = lambda C: LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=SEED)
        self.m_lab = mk(8).fit(X, d["y"]); self.m_act = mk(8).fit(X, d["act"])
        self.m_tech = [mk(4).fit(X, TBd[:, j]) for j in range(6)]
        return self
    def predict(self, texts, bs=50000):
        out = [[], [], []]
        for i in range(0, len(texts), bs):
            t = list(texts[i:i + bs])
            X = hstack([self.cv.transform(t), self.wv.transform(t)]).tocsr()
            for k, m in enumerate([self.m_lab, self.m_act]):
                p = np.zeros((X.shape[0], 6), np.float32); p[:, m.classes_] = m.predict_proba(X); out[k].append(p)
            out[2].append(np.stack([m.predict_proba(X)[:, 1] for m in self.m_tech], 1).astype(np.float32))
        return tuple(np.concatenate(o) for o in out)

def run_cv(model_factory, name, split_list):
    """OOF probabilities averaged over every split where a row is in validation (86x6 for A, C and B)."""
    acc = [np.zeros((len(train), 6), np.float64) for _ in range(3)]; acc_noisy = np.zeros((len(train), 6)); cnt = np.zeros(len(train))
    t0 = time.time()
    for f, (tr_i, va_i) in enumerate(split_list):
        d, TBd = build_train(tr_i, block=blocked_templates(va_i))
        m = model_factory().fit(d, TBd)
        for k, p in enumerate(m.predict(train["n"].values[va_i])): acc[k][va_i] += p
        acc_noisy[va_i] += m.predict(train["n_noisy"].values[va_i])[0]; cnt[va_i] += 1
        if hasattr(m, "free"): m.free()
        if f % 5 == 4: print(f"  {name} repeat {f // 5} done ({time.time() - t0:.0f}s)")
    c = cnt[:, None]
    return [(a / c).astype(np.float32) for a in acc], (acc_noisy / c).astype(np.float32)

OOF = {}
OOF["tfidf"] = run_cv(TfidfModel, "tfidf", splits)
print("TF-IDF (synthetic + real) CV:", scores(train.y.values, train.act.values, TB_real, *OOF["tfidf"][0], PL_noisy=OOF["tfidf"][1]))

# generalization check: train ONLY on synthetic templates that resemble no real row, score on all 86 real rows
_d = synth[~synth["tid"].isin(set(np.where(TEMPLATE_NEAR_REAL)[0]))][["n", "y", "act", "techniques"]]; _m = TfidfModel().fit(_d, np.array([tech_to_vec(s) for s in _d["techniques"]]))
print("TF-IDF synthetic-only -> real:", scores(train.y.values, train.act.values, TB_real, *_m.predict(train["n"].values)))

## 4. Model 2: fine-tuned XLM-RoBERTa (multi-task: label + action + techniques)

In [ ]:
if USE_XLMR:
    import torch, torch.nn as nn, torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AMP = DEVICE.type == "cuda"
    TOK = AutoTokenizer.from_pretrained(MODEL_NAME)

    def set_seed(s):
        random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

    class MultiTask(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = AutoModel.from_pretrained(MODEL_NAME)
            h = self.enc.config.hidden_size
            self.drop = nn.Dropout(0.1)
            self.head_lab, self.head_act, self.head_tech = nn.Linear(h, 6), nn.Linear(h, 6), nn.Linear(h, 6)
        def forward(self, ids, mask):
            x = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
            m = mask.unsqueeze(-1).to(x.dtype)
            x = self.drop((x * m).sum(1) / m.sum(1).clamp(min=1))          # mean pooling
            return self.head_lab(x), self.head_act(x), self.head_tech(x)

    def encode(texts):
        return TOK(list(texts), truncation=True, max_length=MAX_LEN)["input_ids"]

    def collate(ids_list):
        L = max(len(x) for x in ids_list)
        ids = torch.full((len(ids_list), L), TOK.pad_token_id, dtype=torch.long); mask = torch.zeros((len(ids_list), L), dtype=torch.long)
        for i, x in enumerate(ids_list): ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
        return ids.to(DEVICE), mask.to(DEVICE)

    class XlmrModel:
        def fit(self, d, TBd, seed=None):
            seed = SEED if seed is None else seed
            set_seed(seed)
            self.net = MultiTask().to(DEVICE)
            ids = encode(d["n"].values)
            yl = torch.tensor(d["y"].values); ya = torch.tensor(d["act"].values); yt = torch.tensor(TBd, dtype=torch.float32)
            # class weights (inverse sqrt frequency) so the rarer classes are not ignored
            cw = torch.tensor((1 / np.sqrt(np.bincount(d["y"].values, minlength=6) + 1)), dtype=torch.float32); cw = (cw / cw.mean()).to(DEVICE)
            n = len(ids); steps = int(math.ceil(n / BATCH) * EPOCHS)
            no_decay = ["bias", "LayerNorm.weight"]
            params = [{"params": [p for nm, p in self.net.named_parameters() if not any(k in nm for k in no_decay)], "weight_decay": 0.01},
                      {"params": [p for nm, p in self.net.named_parameters() if any(k in nm for k in no_decay)], "weight_decay": 0.0}]
            opt = torch.optim.AdamW(params, lr=LR)
            sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
            scaler = torch.cuda.amp.GradScaler(enabled=AMP)
            g = np.random.RandomState(seed); step = 0; self.net.train(); t0 = time.time()
            while step < steps:
                for b in np.array_split(g.permutation(n), max(1, n // BATCH)):
                    if step >= steps: break
                    x, m = collate([ids[i] for i in b])
                    with torch.autocast(device_type=DEVICE.type, enabled=AMP, dtype=torch.float16):
                        ol, oa, ot = self.net(x, m)
                    ol, oa, ot = ol.float(), oa.float(), ot.float()
                    loss = (F.cross_entropy(ol, yl[b].to(DEVICE), weight=cw, label_smoothing=0.05)
                            + 0.5 * F.cross_entropy(oa, ya[b].to(DEVICE), label_smoothing=0.05)
                            + 0.5 * F.binary_cross_entropy_with_logits(ot, yt[b].to(DEVICE)))
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); sch.step(); step += 1
                    if step % 200 == 0: print(f"    step {step}/{steps} loss {loss.item():.4f} ({time.time() - t0:.0f}s)")
            return self

        @torch.no_grad()
        def predict(self, texts, bs=256):
            self.net.eval()
            ids = encode(texts); order = np.argsort([len(x) for x in ids])    # length-sorted batches = fast inference
            out = [np.zeros((len(ids), 6), np.float32) for _ in range(3)]
            for i in range(0, len(ids), bs):
                b = order[i:i + bs]; x, m = collate([ids[j] for j in b])
                with torch.autocast(device_type=DEVICE.type, enabled=AMP, dtype=torch.float16):
                    ol, oa, ot = self.net(x, m)
                out[0][b] = F.softmax(ol.float(), -1).cpu().numpy(); out[1][b] = F.softmax(oa.float(), -1).cpu().numpy()
                out[2][b] = torch.sigmoid(ot.float()).cpu().numpy()
            return tuple(out)

        def free(self):
            del self.net; import gc; gc.collect(); torch.cuda.empty_cache() if AMP else None

    if RUN_CV:
        # XLM-R: first XLMR_REPEATS repeat(s) of the shared splits (5 fits per repeat); 25 fits would take too long on a T4
        OOF["xlmr"] = run_cv(XlmrModel, "xlmr", splits[:5 * XLMR_REPEATS])
        print("XLM-R CV:", scores(train.y.values, train.act.values, TB_real, *OOF["xlmr"][0], PL_noisy=OOF["xlmr"][1]))

## 5. Ensemble tuning on out-of-fold predictions

In [ ]:
def blend(w, a, b):
    return tuple(w * x + (1 - w) * y for x, y in zip(a, b))

have_x = "xlmr" in OOF
best = None
for w in (np.round(np.arange(0, 1.01, 0.1), 2) if have_x else [0.0]):
    P = blend(w, OOF["xlmr"][0], OOF["tfidf"][0]) if have_x else OOF["tfidf"][0]
    Pn = (w * OOF["xlmr"][1] + (1 - w) * OOF["tfidf"][1]) if have_x else OOF["tfidf"][1]
    for thr in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
        s = scores(train.y.values, train.act.values, TB_real, *P, thr=thr, PL_noisy=Pn)
        if best is None or s["S_proxy"] > best[0]["S_proxy"]: best = (s, w, thr)
_, W_X, THR = best
P_oof = blend(W_X, OOF["xlmr"][0], OOF["tfidf"][0]) if have_x else OOF["tfidf"][0]
P_oof_noisy = (W_X * OOF["xlmr"][1] + (1 - W_X) * OOF["tfidf"][1]) if have_x else OOF["tfidf"][1]
# confidence temperature: minimise the Brier score of "prediction is correct" on OOF
_corr = (P_oof[0].argmax(1) == train.y.values)
TEMP = min(np.arange(0.5, 4.01, 0.05), key=lambda t: np.mean((softmax_t(P_oof[0], t).max(1) - _corr) ** 2))
FINAL_CV = scores(train.y.values, train.act.values, TB_real, *P_oof, thr=THR, PL_noisy=P_oof_noisy, temp=TEMP)
print(f"chosen: xlmr weight={W_X}  technique threshold={THR}  temperature={TEMP:.2f}")
for k in OOF: print(f"  {k:6s} alone:", scores(train.y.values, train.act.values, TB_real, *OOF[k][0], PL_noisy=OOF[k][1]))
print("  ENSEMBLE  :", FINAL_CV)


## 6. Final training on all labeled + synthetic data, test inference, submission

In [ ]:
uniq = test["n"].drop_duplicates().values
inv = pd.Series(np.arange(len(uniq)), index=uniq).loc[test["n"].values].values
print(f"test rows {len(test)} | unique normalized texts {len(uniq)}")
d_all, TB_all = build_train(np.arange(len(train)))

def fit_predict_all(d, TBd, tag):
    t0 = time.time()
    tf = TfidfModel().fit(d, TBd); Pt = tf.predict(uniq); del tf
    print(f"  [{tag}] tfidf done ({time.time() - t0:.0f}s)")
    if not USE_XLMR: return Pt, Pt, Pt
    Pxs = []
    for s in range(FINAL_SEEDS):                 # average several XLM-R seeds for steadier test predictions
        xm = XlmrModel().fit(d, TBd, seed=SEED + s); Pxs.append(xm.predict(uniq)); xm.free()
    Px = tuple(np.mean([p[k] for p in Pxs], axis=0) for k in range(len(Pxs[0])))
    print(f"  [{tag}] xlmr done ({time.time() - t0:.0f}s)")
    return blend(W_X, Px, Pt), Pt, Px

P_test, Pt, Px = fit_predict_all(d_all, TB_all, "final")

lab, act, tb = decode(*P_test, thr=THR)
conf = np.clip(softmax_t(P_test[0], TEMP).max(1), 0.01, 0.99)
pred = pd.DataFrame({"n": uniq, "label": lab, "action": act, "confidence": conf.round(4),
                     "techniques": ["|".join(t for t, v in zip(TECH, r) if v) or "NONE" for r in tb]})
rowwise = pred.iloc[inv].reset_index(drop=True)
sub = pd.DataFrame({"id": test["id"].values, "label": rowwise["label"].astype(int).values, "techniques": rowwise["techniques"].values,
                    "action": rowwise["action"].astype(int).values, "confidence": rowwise["confidence"].values})
sub = sample_sub[["id"]].merge(sub, on="id", how="left")[["id", "label", "techniques", "action", "confidence"]]
sub.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)

# ---- team ensemble artifacts (Rabib's format): oof_<name>_A/B/C.npy (86x6), test_<name>_A/B/C.npy (500000x6) ----
# A, C = class probabilities (cols 0-5); B = T1..T6 probabilities. Test rows follow test.csv order.
# OOF = shared-split CV; test = final models trained on synthetic + real rows only.
def save_team(name, oof3, test3):
    for k, part in enumerate("ABC"):
        key = [0, 2, 1][k]                     # our tuple order is (label, action, tech)
        if oof3 is not None: np.save(os.path.join(OUT_DIR, f"oof_{name}_{part}.npy"), oof3[key].astype(np.float32))
        np.save(os.path.join(OUT_DIR, f"test_{name}_{part}.npy"), test3[key][inv].astype(np.float32))
save_team("tfidf", OOF["tfidf"][0], Pt)
if USE_XLMR:
    save_team("xlmr", OOF["xlmr"][0] if "xlmr" in OOF else None, Px)
save_team("v2ens", P_oof, P_test)
print("saved:", sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".npy")))

# ---- format validation ----
chk = pd.read_csv(os.path.join(OUT_DIR, "submission.csv"), keep_default_na=False)
assert list(chk.columns) == ["id", "label", "techniques", "action", "confidence"]
assert len(chk) == len(test) and chk["id"].is_unique and set(chk["id"]) == set(test["id"])
assert chk["label"].between(0, 5).all() and chk["action"].between(0, 5).all() and chk["confidence"].between(0, 1).all()
assert chk["techniques"].map(lambda s: s == "NONE" or set(s.split("|")) <= set(TECH)).all()
print("OK submission.csv:", chk.shape)
print("test label mix:", chk["label"].value_counts(normalize=True).sort_index().round(3).to_dict())
print("test action mix:", chk["action"].value_counts(normalize=True).sort_index().round(3).to_dict())
print("confidence mean:", round(float(chk["confidence"].mean()), 4), "| NONE share:", round(float((chk["techniques"] == "NONE").mean()), 4))

## 7. Error analysis (out-of-fold, real labeled rows)

In [ ]:
oof_lab = decode(*P_oof, thr=THR)[0]
print("confusion matrix (rows = true, cols = predicted):\n", confusion_matrix(train.y.values, oof_lab, labels=range(6)))
print(classification_report(train.y.values, oof_lab, labels=range(6), digits=3, zero_division=0))
for lg, g in train.groupby("lang"):
    print(f"lang {lg:9s} n={len(g):3d}  acc={(oof_lab[g.index] == g.y.values).mean():.3f}")
conv = train["text"].str.contains(r"\|\|\|")
print(f"conversations n={conv.sum()} acc={(oof_lab[conv.values] == train.y.values[conv.values]).mean():.3f} | "
      f"single n={(~conv).sum()} acc={(oof_lab[~conv.values] == train.y.values[~conv.values]).mean():.3f}")
err = train.assign(pred=oof_lab, p_true=P_oof[0][np.arange(len(train)), train.y.values])
print("\nlowest-confidence-on-truth rows:")
print(err.sort_values("p_true").head(10)[["y", "pred", "p_true", "text"]].to_string(max_colwidth=100))
print("\nSUMMARY  CV proxy:", FINAL_CV)

# Part B — Embedding family (Rabib)
## B1. Sentence embeddings: multilingual-e5-large, bge-m3, LaBSE (GPU, about 47 min)

In [ ]:
import os, re, sys, time, subprocess, unicodedata
import numpy as np, pandas as pd, torch
try:
    import sentence_transformers
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    import sentence_transformers
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'GPU is OFF: Settings -> Accelerator -> GPU T4 x2'
print('sentence-transformers', sentence_transformers.__version__, '| torch', torch.__version__)

D = next(p for p in ['/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/', '/kaggle/input/nsucec-neurahack-2026-preliminary/'] if __import__('os').path.exists(p + 'test.csv'))
OUT = '/kaggle/working/'
train = pd.read_csv(D + 'sample.csv')
test  = pd.read_csv(D + 'test.csv')

# ---- shared cleaning (give this function to Anadi + Samprity) ----
LOOK = str.maketrans({
    'а':'a','е':'e','о':'o','р':'p','с':'c','у':'y','х':'x','і':'i','ј':'j','ѕ':'s','к':'k','м':'m','т':'t','н':'h','в':'b',
    'А':'A','В':'B','Е':'E','К':'K','М':'M','Н':'H','О':'O','Р':'P','С':'C','Т':'T','Х':'X','І':'I',
    'α':'a','ο':'o','ε':'e','ν':'v','ρ':'p','τ':'t','ι':'i','κ':'k','υ':'u','Ο':'O','Α':'A','Ε':'E','Ρ':'P','Τ':'T','Ι':'I','Κ':'K','Ν':'N'})
ZW = re.compile('[\u200b-\u200f\u2060\ufeff\u00ad]')
def clean(t):
    t = unicodedata.normalize('NFKC', str(t))
    t = ZW.sub('', t).translate(LOOK).lower()
    t = re.sub(r'hxxps?', 'http', t)
    t = t.replace('[.]', '.').replace('(.)', '.').replace('[dot]', '.')
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)          # "piiin" -> "piin"
    t = t.replace('|||', ' \n ')
    t = re.sub(r'[ \t]+', ' ', t).strip()
    return t

tr_txt = train['text'].map(clean).tolist()
te_txt = test['text'].map(clean).tolist()
print(tr_txt[:3])

MODELS = {
    'e5':    ('intfloat/multilingual-e5-large', 'query: '),
    'bge':   ('BAAI/bge-m3', ''),
    'labse': ('sentence-transformers/LaBSE', ''),
}

def run(key, bs=256):
    name, pre = MODELS[key]
    p_tr, p_te = f'{OUT}emb_{key}_train.npy', f'{OUT}emb_{key}_test.npy'
    if os.path.exists(p_tr) and os.path.exists(p_te):
        print(key, 'cached'); return
    m = SentenceTransformer(name, device='cuda'); m.half(); m.max_seq_length = 256
    enc = lambda xs: m.encode([pre + x for x in xs], batch_size=bs, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False)
    rng = np.random.RandomState(SEED)
    probe = [te_txt[i] for i in rng.choice(len(te_txt), 5000, replace=False)]
    t0 = time.time(); enc(probe); r = 5000 / (time.time() - t0)
    print(f'{key}: {r:.0f} rows/s -> full test ~{len(te_txt)/r/60:.1f} min')
    np.save(p_tr, enc(tr_txt).astype(np.float16))
    t0 = time.time()
    E = m.encode([pre + x for x in te_txt], batch_size=bs, normalize_embeddings=True,
                 convert_to_numpy=True, show_progress_bar=True)
    np.save(p_te, E.astype(np.float16))
    print(key, 'done', E.shape, f'{(time.time()-t0)/60:.1f} min')
    del m, E; torch.cuda.empty_cache()

for k in ['e5', 'bge', 'labse']:
    run(k)
print(os.listdir(OUT))

## B2. Classifier heads on the shared folds (`folds_5x5.npy`), saves `oof_/test_{e5,bge,labse,emb}_{A,B,C}.npy`

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import f1_score
SEED = 42
D = next(p for p in ['/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/', '/kaggle/input/nsucec-neurahack-2026-preliminary/'] if __import__('os').path.exists(p + 'test.csv')); OUT = '/kaggle/working/'
train = pd.read_csv(D + 'sample.csv'); n = len(train)
yA = train['label'].values; yC = train['action'].values
T = [f'T{i}' for i in range(1, 7)]
YB = np.array([[int(t in s) for t in T] for s in train['techniques'].fillna('NONE').astype(str).str.split('|')])

# shared folds: 5 repeats x 5 folds, stratified on label (send folds_5x5.npy to Anadi + Samprity)
splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED).split(np.zeros(n), yA))
folds = np.zeros((5, n), int)
for i, (_, va) in enumerate(splits): folds[i // 5, va] = i % 5
np.save(OUT + 'folds_5x5.npy', folds)

def lr(): return LogisticRegression(C=4.0, class_weight='balanced', max_iter=3000)
def proba6(m, X):
    P = np.zeros((len(X), 6)); P[:, m.classes_] = m.predict_proba(X); return P
def fitB(X, Y):
    return [lr().fit(X, Y[:, j]) if 0 < Y[:, j].sum() < len(Y) else float(Y[:, j].mean()) for j in range(6)]
def predB(ms, X):
    return np.column_stack([m.predict_proba(X)[:, 1] if not isinstance(m, float) else np.full(len(X), m) for m in ms])

def post(PA, PB, PC, thr=0.5):
    a = PA.argmax(1); fraud = np.isin(a, [1, 2, 3, 4])
    B = (PB >= thr).astype(int); B[~fraud] = 0
    idx = np.where(fraud & (B.sum(1) == 0))[0]; B[idx, PB[idx].argmax(1)] = 1
    C = PC.copy(); C[fraud, 0] = -1; act = C.argmax(1); act[~fraud] = 0
    return a, B, act
def score(PA, PB, PC):
    a, B, act = post(PA, PB, PC)
    fa = f1_score(yA, a, average='macro')
    fb = f1_score(YB, B, average='macro', zero_division=0)
    fc = f1_score(yC, act, average='macro', zero_division=0)
    return fa, fb, fc, (0.40 * fa + 0.15 * fb + 0.10 * fc) / 0.65

def report(name, oA, oB, oC):
    S = np.array([score(oA[r], oB[r], oC[r]) for r in range(5)])
    m, s = S.mean(0), S.std(0)
    print(f'{name:6s} A {m[0]:.3f}±{s[0]:.3f} | B {m[1]:.3f}±{s[1]:.3f} | C {m[2]:.3f}±{s[2]:.3f} | proxy {m[3]:.3f}')

keys = [k for k in ['e5', 'bge', 'labse'] if os.path.exists(f'{OUT}emb_{k}_test.npy')]
OOF = {}
for k in keys:
    X = np.load(f'{OUT}emb_{k}_train.npy').astype(np.float32)
    oA, oB, oC = (np.zeros((5, n, 6)) for _ in range(3))
    for i, (tr, va) in enumerate(splits):
        r = i // 5
        oA[r, va] = proba6(lr().fit(X[tr], yA[tr]), X[va])
        oB[r, va] = predB(fitB(X[tr], YB[tr]), X[va])
        oC[r, va] = proba6(lr().fit(X[tr], yC[tr]), X[va])
    OOF[k] = (oA, oB, oC); report(k, oA, oB, oC)
    # full fit -> test, chunked
    mA, mB, mC = lr().fit(X, yA), fitB(X, YB), lr().fit(X, yC)
    E = np.load(f'{OUT}emb_{k}_test.npy', mmap_mode='r')
    tA, tB, tC = [], [], []
    for s in range(0, len(E), 50000):
        Xt = np.asarray(E[s:s + 50000], dtype=np.float32)
        tA.append(proba6(mA, Xt)); tB.append(predB(mB, Xt)); tC.append(proba6(mC, Xt))
    for t, oo, tt in zip('ABC', (oA, oB, oC), (tA, tB, tC)):
        np.save(f'{OUT}oof_{k}_{t}.npy', oo.mean(0)); np.save(f'{OUT}test_{k}_{t}.npy', np.vstack(tt))

# simple average of the 3 embedding models = "emb" family
avg = [np.mean([OOF[k][j] for k in keys], 0) for j in range(3)]
report('emb', *avg)
for j, t in enumerate('ABC'):
    np.save(f'{OUT}oof_emb_{t}.npy', avg[j].mean(0))
    np.save(f'{OUT}test_emb_{t}.npy', np.mean([np.load(f'{OUT}test_{k}_{t}.npy') for k in keys], 0))
print('saved:', sorted(f for f in os.listdir(OUT) if f.startswith(('oof_', 'test_', 'folds'))))

## B3. Generalization and robustness analysis (clean vs noisy, language, conversation, hard legitimate segments)

In [ ]:
import re, unicodedata, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
SEED = 42; rng = np.random.RandomState(SEED); torch.manual_seed(SEED)
D = next(p for p in ['/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/', '/kaggle/input/nsucec-neurahack-2026-preliminary/'] if __import__('os').path.exists(p + 'test.csv')); OUT = '/kaggle/working/'
train = pd.read_csv(D + 'sample.csv'); n = len(train); txt = train['text'].astype(str)
yA = train['label'].values; yC = train['action'].values
TC = [f'T{i}' for i in range(1, 7)]
YB = np.array([[int(t in s) for t in TC] for s in train['techniques'].fillna('NONE').astype(str).str.split('|')])
KEYS = ['e5', 'bge', 'labse']; W = np.array([0.367, 0.417, 0.217]); THR = 0.45
MODELS = {'e5': ('intfloat/multilingual-e5-large', 'query: '), 'bge': ('BAAI/bge-m3', ''),
          'labse': ('sentence-transformers/LaBSE', '')}

# same clean() as the embedding cell
LOOK = str.maketrans({
    'а':'a','е':'e','о':'o','р':'p','с':'c','у':'y','х':'x','і':'i','ј':'j','ѕ':'s','к':'k','м':'m','т':'t','н':'h','в':'b',
    'А':'A','В':'B','Е':'E','К':'K','М':'M','Н':'H','О':'O','Р':'P','С':'C','Т':'T','Х':'X','І':'I',
    'α':'a','ο':'o','ε':'e','ν':'v','ρ':'p','τ':'t','ι':'i','κ':'k','υ':'u','Ο':'O','Α':'A','Ε':'E','Ρ':'P','Τ':'T','Ι':'I','Κ':'K','Ν':'N'})
ZW = re.compile('[\u200b-\u200f\u2060\ufeff\u00ad]')
def clean(t):
    t = unicodedata.normalize('NFKC', str(t)); t = ZW.sub('', t).translate(LOOK).lower()
    t = re.sub(r'hxxps?', 'http', t); t = t.replace('[.]', '.').replace('(.)', '.').replace('[dot]', '.')
    t = re.sub(r'(.)\1{2,}', r'\1\1', t); t = t.replace('|||', ' \n ')
    return re.sub(r'[ \t]+', ' ', t).strip()

# ---- adversarial noise like Section 7 of the problem statement ----
LOOK_REV = {'a': 'а', 'e': 'е', 'o': 'о', 'p': 'р', 'c': 'с', 'x': 'х', 'i': 'і'}
def noise(t, r):
    out = []
    for ch in str(t):
        u = r.rand()
        if ch.isalpha() and u < 0.04: continue                                  # delete
        if ch.isalpha() and u < 0.08: out.append(ch * 3); continue              # repeat
        if ch.lower() in LOOK_REV and u < 0.14: out.append(LOOK_REV[ch.lower()]); continue  # lookalike
        if ch == ' ' and u < 0.25: continue                                     # missing space
        out.append(ch.upper() if r.rand() < 0.15 else ch)                       # random caps
    s = ''.join(out)
    if r.rand() < 0.5:                                                          # junk string
        junk = ''.join(r.choice(list('qwrtzxkjvbnm'), r.randint(6, 12))); p = r.randint(0, len(s) + 1)
        s = s[:p] + ' ' + junk + ' ' + s[p:]
    return s
noisy = [noise(t, rng) for t in txt]
for i in range(3): print('clean:', txt[i], '\nnoisy:', noisy[i], '\n')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
Xn = {}
for k, (name, pre) in MODELS.items():
    m = SentenceTransformer(name, device=dev); m.max_seq_length = 256
    Xn[k] = m.encode([pre + clean(t) for t in noisy], batch_size=16, normalize_embeddings=True).astype(np.float32)
    del m; print(k, 'noisy encoded')
Xtr = {k: np.load(f'{OUT}emb_{k}_train.npy').astype(np.float32) for k in KEYS}

def lr(): return LogisticRegression(C=4.0, class_weight='balanced', max_iter=3000)
def proba6(m, X):
    P = np.zeros((len(X), 6)); P[:, m.classes_] = m.predict_proba(X); return P
def fitB(X, Y): return [lr().fit(X, Y[:, j]) if 0 < Y[:, j].sum() < len(Y) else float(Y[:, j].mean()) for j in range(6)]
def predB(ms, X): return np.column_stack([m.predict_proba(X)[:, 1] if not isinstance(m, float) else np.full(len(X), m) for m in ms])
def fit3(X, a, B, c): return (lr().fit(X, a), fitB(X, B), lr().fit(X, c))
def pred3(m, X): return proba6(m[0], X), predB(m[1], X), proba6(m[2], X)
def post(PA, PB, PC, thr=THR):
    a = PA.argmax(1); fraud = np.isin(a, [1, 2, 3, 4])
    B = (PB >= thr).astype(int); B[~fraud] = 0
    idx = np.where(fraud & (B.sum(1) == 0))[0]; B[idx, PB[idx].argmax(1)] = 1
    C = PC.copy(); C[fraud, 0] = -1; act = C.argmax(1); act[~fraud] = 0
    return a, B, act
def scores(a, B, act):
    fa = f1_score(yA, a, average='macro'); fb = f1_score(YB, B, average='macro', zero_division=0)
    fc = f1_score(yC, act, average='macro', zero_division=0)
    return np.round([(0.40 * fa + 0.15 * fb + 0.10 * fc) / 0.65, fa, fb, fc], 3)

# ---- same folds: train on clean, predict clean vs noisy validation rows ----
folds = np.load(OUT + 'folds_5x5.npy'); R = 2
oc = {k: [np.zeros((R, n, 6)) for _ in range(3)] for k in KEYS}; on = {k: [np.zeros((R, n, 6)) for _ in range(3)] for k in KEYS}
for r in range(R):
    for f in range(5):
        va, tr = np.where(folds[r] == f)[0], np.where(folds[r] != f)[0]
        for k in KEYS:
            m = fit3(Xtr[k][tr], yA[tr], YB[tr], yC[tr])
            o1, o2 = pred3(m, Xtr[k][va]), pred3(m, Xn[k][va])
            for j in range(3): oc[k][j][r, va] = o1[j]; on[k][j][r, va] = o2[j]
def ens(d): return [sum(W[i] * d[k][j].mean(0) for i, k in enumerate(KEYS)) for j in range(3)]
ac, Bc, cc = post(*ens(oc)); an, Bn, cn = post(*ens(on))
print('\n== ROBUSTNESS (proxy / A / B / C) ==')
print('clean:', scores(ac, Bc, cc)); print('noisy:', scores(an, Bn, cn))

# ---- segments ----
BANGLISH = set('ami apni apnar tomar amar korun koren korse kore din dao taka ase ache hobe hoise bhai mama ekhon ajke '
               'bolen pathaisi ferot jitechen kichu lagbe bari asbo nai jan atke dite korte bose ghor protek korle'.split())
def lang(t):
    if re.search(r'[\u0980-\u09FF]', t): return 'mixed' if re.search(r'[A-Za-z]{3,}', t) else 'bangla'
    return 'banglish' if any(w in BANGLISH for w in re.findall(r'[a-z]+', t.lower())) else 'english'
KW = r'otp|pin|link|http|parcel|deliver|cashback|offer|account|payment|taka|tk|bkash|nagad|code'
segs = {'language': txt.map(lang).values,
        'format': np.where(txt.str.contains(r'\|\|\|', regex=True), 'conversation', 'single'),
        'hard_legit': np.where((yA == 0) & txt.str.lower().str.contains(KW), 'hard legit', np.where(yA == 0, 'easy legit', 'non-legit'))}
rows = []
for sname, seg in segs.items():
    for v in pd.unique(seg):
        mk = seg == v
        rows.append([sname, v, int(mk.sum()), (ac[mk] == yA[mk]).mean(), (an[mk] == yA[mk]).mean(), (cc[mk] == yC[mk]).mean()])
rep = pd.DataFrame(rows, columns=['segment', 'value', 'n', 'acc_A_clean', 'acc_A_noisy', 'acc_C_clean']).round(3)
print('\n== SEGMENTS ==\n', rep.to_string(index=False))

names = ['Legit', 'MFS/OTP', 'Prize', 'Job/Inv', 'Parcel', 'Promo']
print('\n== per-class F1 (A, clean) ==', dict(zip(names, f1_score(yA, ac, average=None).round(3))))
print('== per-technique F1 (B, clean) ==', dict(zip(TC, f1_score(YB, Bc, average=None, zero_division=0).round(3))))
print('\n== confusion A (rows=true, cols=pred) ==')
print(pd.DataFrame(confusion_matrix(yA, ac, labels=range(6)), index=names, columns=names))
print('legit predicted as fraud:', round(np.isin(ac[yA == 0], [1, 2, 3, 4]).mean(), 3),
      '| promo predicted as fraud:', round(np.isin(ac[yA == 5], [1, 2, 3, 4]).mean(), 3))
rep.to_csv(OUT + 'generalization_report.csv', index=False)

# Part C — Final ensemble of all families, writes `submission.csv`
Noise augmentation of the real rows for the embedding family was tested and rejected by CV (see write-up). No family uses test text for training.

In [ ]:
import numpy as np, pandas as pd, itertools, os
from sklearn.metrics import f1_score
SEED = 42
D = next(p for p in ['/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/', '/kaggle/input/nsucec-neurahack-2026-preliminary/'] if __import__('os').path.exists(p + 'test.csv')); OUT = '/kaggle/working/'
train = pd.read_csv(D + 'sample.csv'); test = pd.read_csv(D + 'test.csv', usecols=['id'])
ss_cols = list(pd.read_csv(D + 'sample_submission.csv', nrows=1).columns)
n, NT = len(train), len(test)
yA = train['label'].values; yC = train['action'].values
TC = [f'T{i}' for i in range(1, 7)]
YB = np.array([[int(t in s) for t in TC] for s in train['techniques'].fillna('NONE').astype(str).str.split('|')])

USE_PL = False    # set False if Cell 5 showed 'pl' worse than 'base'
EMB = [k + ('pl' if USE_PL else '') for k in ['e5', 'bge', 'labse']]
EMB_W = np.array([0.367, 0.417, 0.217])

def load(name, split):
    return [np.load(f'{OUT}{split}_{name}_{t}.npy').astype(np.float32) for t in 'ABC']
def emb_family(split):
    P = [load(k, split) for k in EMB]
    return [sum(EMB_W[i] * P[i][j] for i in range(3)) for j in range(3)]

# ---- collect families that exist + sanity checks ----
O, TS = {'emb': emb_family('oof')}, {'emb': emb_family('test')}
for name in ['tfidf', 'xlmr', 'ft']:   # Anadi: tfidf + xlmr (v2 cells above); Samprity: ft
    if all(os.path.exists(f'{OUT}{s}_{name}_{t}.npy') for s in ['oof', 'test'] for t in 'ABC'):
        O[name], TS[name] = load(name, 'oof'), load(name, 'test')
NAMES = list(O)
for k in NAMES:
    for j, t in enumerate('ABC'):
        assert O[k][j].shape == (n, 6) and TS[k][j].shape == (NT, 6), f'{k}_{t} wrong shape'
        assert np.isfinite(O[k][j]).all() and np.isfinite(TS[k][j]).all(), f'{k}_{t} has NaN'
        assert O[k][j].min() >= 0 and O[k][j].max() <= 1, f'{k}_{t} not probabilities'
print('families:', NAMES)

def post(PA, PB, PC, thr):
    a = PA.argmax(1); fraud = np.isin(a, [1, 2, 3, 4])
    B = (PB >= thr).astype(int); B[~fraud] = 0
    idx = np.where(fraud & (B.sum(1) == 0))[0]; B[idx, PB[idx].argmax(1)] = 1
    C = PC.copy(); C[fraud, 0] = -1; act = C.argmax(1); act[~fraud] = 0
    return a, B, act
def proxy(PA, PB, PC, thr):
    a, B, act = post(PA, PB, PC, thr)
    fa = f1_score(yA, a, average='macro'); fb = f1_score(YB, B, average='macro', zero_division=0)
    fc = f1_score(yC, act, average='macro', zero_division=0)
    return np.array([(0.40 * fa + 0.15 * fb + 0.10 * fc) / 0.65, fa, fb, fc])
def mix(w, src): return [sum(w[i] * src[k][j] for i, k in enumerate(NAMES)) for j in range(3)]

for k in NAMES:   # each family alone; a score near 1.0 means that family leaked labels
    print(f'{k:6s} proxy/A/B/C', proxy(*O[k], 0.45).round(3))

# ---- weights + threshold by CV, shrunk 50% toward equal ----
grid = [w for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=len(NAMES)) if abs(sum(w) - 1) < 1e-6]
res = sorted(((proxy(*mix(w, O), thr)[0], tuple(np.round(w, 1)), round(thr, 2))
              for w in grid for thr in np.arange(0.3, 0.71, 0.05)), reverse=True)
print('top 3:', res[:3])
w = 0.5 * np.array(res[0][1]) + 0.5 / len(NAMES); thr = res[0][2]
print('weights', dict(zip(NAMES, w.round(3))), 'thr', thr, '| CV proxy/A/B/C', proxy(*mix(w, O), thr).round(3))

# ---- candidates: shrunk ensemble vs. the TF-IDF + XLM-R blend from Part A (v2ens); keep the better CV proxy ----
CANDS = {'ens_' + '_'.join(NAMES): (mix(w, O), lambda: mix(w, TS), thr)}
if all(os.path.exists(f'{OUT}{s}_v2ens_{t}.npy') for s in ['oof', 'test'] for t in 'ABC'):
    Ov = load('v2ens', 'oof')
    thr_v = max(np.round(np.arange(0.3, 0.71, 0.05), 2), key=lambda t: proxy(*Ov, t)[0])
    CANDS['v2ens'] = (Ov, lambda: load('v2ens', 'test'), thr_v)
SCORES = {k: proxy(*v[0], v[2]) for k, v in CANDS.items()}
for k, s in SCORES.items(): print(f'{k:22s} thr {CANDS[k][2]} | CV proxy/A/B/C', s.round(3))
WIN = max(SCORES, key=lambda k: SCORES[k][0]); print('chosen for submission.csv:', WIN)

# ---- confidence: temperature by Brier, then test -> csv (one file per candidate) ----
def conf_of(P, tau):
    L = np.log(np.clip(P, 1e-9, 1)) / tau; L -= L.max(1, keepdims=True)
    S = np.exp(L); S /= S.sum(1, keepdims=True); return S.max(1)
for k, (Oc, Tc, th) in CANDS.items():
    PA, PB, PC = Oc; PA = PA / PA.sum(1, keepdims=True)
    correct = (post(PA, PB, PC, th)[0] == yA).astype(float)
    briers = {tau: np.mean((conf_of(PA, tau) - correct) ** 2) for tau in np.round(np.arange(0.05, 3.01, 0.05), 2)}
    tau = min(briers, key=briers.get); print(f'{k}: Brier raw {briers[1.0]:.4f} -> tau {tau}: {briers[tau]:.4f}')
    PA, PB, PC = Tc(); PA = PA / PA.sum(1, keepdims=True)
    a, B, act = post(PA, PB, PC, th)
    sub = pd.DataFrame({'id': test['id'].values, 'label': a.astype(int),
                        'techniques': ['|'.join(TC[j] for j in np.flatnonzero(r)) or 'NONE' for r in B],
                        'action': act.astype(int), 'confidence': np.clip(conf_of(PA, tau), 0.02, 0.98).round(4)})[ss_cols]
    assert list(sub.columns) == ['id', 'label', 'techniques', 'action', 'confidence']
    assert len(sub) == NT and sub['id'].is_unique and (sub['id'].values == test['id'].values).all()
    assert sub['label'].between(0, 5).all() and sub['action'].between(0, 5).all()
    assert sub['techniques'].str.fullmatch(r'NONE|T[1-6](\|T[1-6])*').all() and not sub.isna().any().any()
    sub.to_csv(OUT + f'sub_{k}.csv', index=False); print('wrote', f'sub_{k}.csv')
    if k == WIN:
        sub.to_csv(OUT + 'submission.csv', index=False)
        print('submission.csv =', k, '| test label share:', sub['label'].value_counts(normalize=True).sort_index().round(3).to_dict())


# Review

## Why each model
| Family | Why it is in the ensemble | Out-of-fold evidence (86 real rows) |
|---|---|---|
| `tfidf` + synthetic data | Character n-grams survive typos, lookalikes and leetspeak; cheap enough for 25 CV fits; learns the synthetic templates well. | v2.1: Task A 0.877, B 0.752, C 0.710; noisy-text A 0.842. v3 run: Task A 0.903, B 0.774, C 0.767; noisy-text A 0.879 |
| `xlmr` (`xlm-roberta-base`) + synthetic data | A multilingual encoder for Bangla, Banglish and mixed text; one multi-task head shares information between A, B and C. | v3 run: Task A 0.919, B 0.820, C 0.924; noisy-text A 0.918 (1 repeat) |
| `emb` (e5-large, bge-m3, LaBSE, frozen) | Needs no training data beyond the 86 rows, is a strong semantic baseline, and is a different kind of model from the other two families, so its errors are less correlated with theirs. | Alone: e5 0.678, bge-m3 0.755, LaBSE 0.642 proxy; weighted family 0.776 (A 0.841, B 0.737, C 0.576) |
| Final choice | Averages these different kinds of models; weights are shrunk toward equal because 86 rows are too few to trust a sharp optimum. The TF-IDF + XLM-R blend is kept as a second candidate, and the higher CV proxy wins. | v3 run: TF-IDF + XLM-R blend chosen: proxy 0.915, Task A 0.971, B 0.840, C 0.802 (shrunk 3-family ensemble: proxy 0.896, A 0.952, B 0.827, C 0.777). In-sample selection, so slightly optimistic; the v3 templates were also written from these rows' errors. |

## Error and robustness analysis
* **Most common errors (embedding family, clean text):** legitimate messages predicted as fraud (18.8% of class 0; 4 predicted as MFS/OTP, 2 as parcel). Next come MFS/OTP predicted as legitimate (2 of 12) and prize messages predicted as MFS/OTP or promo (1 each of 10). These error types are why the v3 templates were added: legitimate telecom, courier, HR and institution notices; non-scam promotions; prize-with-OTP (class 1).
* **Per-class F1 (emb, A):** Legit 0.833, MFS/OTP 0.714, Prize 0.889, Job/Investment 0.909, Parcel 0.900, Promo 0.833. Per-technique F1 (B): T1 0.667, T2 0.692, T3 0.750, T4 0.857, T5 0.650, T6 0.818.
* **Robustness to adversarial noise (emb):** the proxy drops from 0.781 on clean text to 0.655 on corrupted text (A 0.846 → 0.712). Banglish is hit hardest (accuracy 0.783 → 0.609), and so are conversations (0.875 → 0.625). TF-IDF with noised synthetic rows holds up better (A 0.877 → 0.842), which is one reason both families are kept.
* **Segments (emb, accuracy A clean):** English 0.842 (n=57), Banglish 0.783 (n=23), Bangla 1.000 (n=6); single messages 0.833, conversations 0.875; hard legitimate cases 0.714.
* **Rejected ideas (by CV):**
  * Noise augmentation of the real rows for the embedding family: 0.727 vs 0.756 on 0.8 × clean + 0.2 × noisy.
  * Noise on the real rows for TF-IDF: A 0.877 → 0.865.
  * Pseudo-labeling was also removed because Rulebook v1.3 bans training on test text.
* **Limits:** only 86 labeled rows, so every score has wide error bars (±0.02 across repeats). The composite metric's domain-shift and novel-scenario parts are approximated by Task A F1.

## Speed (Kaggle T4, fp16)
| Step | Time |
|---|---|
| e5-large encoding, 500k rows | 21.1 min (≈ 418 rows/s) |
| bge-m3 encoding, 500k rows | 20.1 min (≈ 409 rows/s) |
| LaBSE encoding, 500k rows | 6.1 min (≈ 1,384 rows/s) |
| Embedding heads (25-split CV + full fit + 500k inference, 3 models) | about 1 min |
| TF-IDF 25-split CV + final fit + 500k inference | about 12 min (CV 9.4 min, final 2.3 min) |
| XLM-R (xlm-roberta-base) 5 CV fits + 2 final seeds + 500k inference | about 51 min (CV 27 min, final 2 seeds + inference 24 min) |
| Whole notebook | 107 min (v3 run) |